In [ ]:
apikey = '56a07e920d1f7f0e9aed6c3bc6a62491c21620c2'

# XBRL 퇴직연금 부채 인덱스 — ELEMENT_ID 패턴 탐색

DB형 퇴직연금 부채 인덱스 산출에 필요한 변수들의 ELEMENT_ID 패턴을 DART XBRL TSV 데이터에서 탐색한다.

**대상**: 2024_4Q (3,000개사) + 2023_4Q (2,800개사)  
**목표**: 변수별 ELEMENT_ID 매핑 전략 수립 (IFRS 표준 / DART 확장 / 기업 자체 확장)

In [10]:
# =============================================================================
# Cell 1: 데이터 기본 현황 (Step 1)
# =============================================================================
import pandas as pd
import numpy as np
import os, gc, re
from pathlib import Path

BASE = Path(r'C:\Users\user\Downloads\python\DB_Index')
QUARTERS = ['2024_4Q', '2023_4Q']

# --- sub.tsv 로드 ---
subs = {}
for q in QUARTERS:
    subs[q] = pd.read_csv(BASE / q / 'sub.tsv', sep='\t', dtype=str)
    print(f"[{q}] sub.tsv: {len(subs[q]):,}개사, REPORT_DATE 분포:")
    print(subs[q]['REPORT_DATE'].value_counts().to_string())
    print()

# 공통 CIK
cik_24 = set(subs['2024_4Q']['CIK'])
cik_23 = set(subs['2023_4Q']['CIK'])
print(f"2024_4Q CIK: {len(cik_24):,}개")
print(f"2023_4Q CIK: {len(cik_23):,}개")
print(f"공통 CIK: {len(cik_24 & cik_23):,}개")
print(f"2024에만: {len(cik_24 - cik_23):,}개, 2023에만: {len(cik_23 - cik_24):,}개")

# --- val.tsv 행 수 & 고유 ELEMENT_ID 미리보기 ---
for q in QUARTERS:
    vpath = BASE / q / 'val.tsv'
    # 행 수는 이미 확인됨, ELEMENT_ID만 빠르게 로드
    elem_col = pd.read_csv(vpath, sep='\t', usecols=['ELEMENT_ID'], dtype=str)
    print(f"\n[{q}] val.tsv: {len(elem_col):,} rows, 고유 ELEMENT_ID: {elem_col['ELEMENT_ID'].nunique():,}개")
    
    # ELEMENT_ID prefix 분포 (상위 10개)
    prefixes = elem_col['ELEMENT_ID'].str.split('_', n=1).str[0]
    print("  ELEMENT_ID prefix 분포 (상위 10):")
    print(prefixes.value_counts().head(10).to_string())
    del elem_col, prefixes
    gc.collect()

[2024_4Q] sub.tsv: 2,965개사, REPORT_DATE 분포:
REPORT_DATE
20241231    2912
20240331      16
20240630      14
20240930      10
20241130       6
20240831       3
20240229       1
20240731       1
20240430       1
20241031       1

[2023_4Q] sub.tsv: 2,826개사, REPORT_DATE 분포:
REPORT_DATE
20231231    2805
20230930      10
20231130       6
20230831       2
20230630       2
20231031       1

2024_4Q CIK: 2,965개
2023_4Q CIK: 2,826개
공통 CIK: 2,740개
2024에만: 225개, 2023에만: 86개

[2024_4Q] val.tsv: 5,546,754 rows, 고유 ELEMENT_ID: 156,971개
  ELEMENT_ID prefix 분포 (상위 10):
ELEMENT_ID
ifrs-full         3135236
dart              1250523
dart-gcd           213883
entity00120030      11015
entity00153861       9980
entity00258801       9444
entity00159193       9276
entity00124540       7601
entity00261285       5926
entity00904672       5328

[2023_4Q] val.tsv: 3,789,744 rows, 고유 ELEMENT_ID: 138,204개
  ELEMENT_ID prefix 분포 (상위 10):
ELEMENT_ID
ifrs-full         1978807
dart               812108
dart-gcd       

In [11]:
# =============================================================================
# Cell 2: val.tsv 전체 로드 + 퇴직연금 키워드 필터 (Step 2-A)
# =============================================================================
import csv

# v2: 22→9개 축소 (IFRS 퇴직연금 EID는 거의 모두 DefinedBenefit/PostemploymentBenefit 포함)
PENSION_KEYWORDS_EN = '|'.join([
    'DefinedBenefit', 'PostemploymentBenefit', 'RetirementBenefit',
    'PlanAsset', 'FairValueOfPlanAsset', 'ActuarialAssumption',
    'BenefitPayment', 'PaymentsFromPlan', 'Severance',
])

val_filtered = {}
for q in QUARTERS:
    print(f"\n{'='*60}")
    print(f"[{q}] val.tsv 로딩 중...")
    vpath = BASE / q / 'val.tsv'
    
    df_val = pd.read_csv(
        vpath, sep='\t',
        usecols=['CIK', 'REPORT_DATE', 'ELEMENT_ID', 'TAXONOMY_ID', 'CONTEXT_ID', 'UNIT_ID', 'DECIMALS', 'VALUE'],
        dtype={'CIK': str, 'REPORT_DATE': str, 'ELEMENT_ID': str,
               'TAXONOMY_ID': str, 'CONTEXT_ID': str, 'UNIT_ID': str,
               'DECIMALS': str, 'VALUE': str},
        quoting=csv.QUOTE_NONE,
        on_bad_lines='warn',
    )
    print(f"  전체 로드: {len(df_val):,} rows, 메모리: {df_val.memory_usage(deep=True).sum()/1e9:.2f} GB")
    
    # 퇴직연금 키워드 필터
    mask = df_val['ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False)
    val_filtered[q] = df_val[mask].copy()
    
    pct = len(val_filtered[q]) / len(df_val) * 100
    print(f"  필터 결과: {len(val_filtered[q]):,} rows ({pct:.2f}%)")
    print(f"  고유 CIK: {val_filtered[q]['CIK'].nunique():,}개")
    print(f"  고유 ELEMENT_ID: {val_filtered[q]['ELEMENT_ID'].nunique():,}개")
    
    # 원본 해제
    del df_val
    gc.collect()

# --- 필터 결과 ELEMENT_ID 분포 ---
for q in QUARTERS:
    print(f"\n[{q}] 필터 ELEMENT_ID 상위 30:")
    print(val_filtered[q]['ELEMENT_ID'].value_counts().head(30).to_string())

# ELEMENT_ID prefix 유형 분류
for q in QUARTERS:
    df = val_filtered[q]
    def classify_prefix(eid):
        if eid.startswith('ifrs-full_'):
            return 'ifrs-full'
        elif eid.startswith('dart_'):
            return 'dart'
        elif eid.startswith('entity'):
            return 'entity (기업확장)'
        else:
            return 'other'
    
    df_types = df['ELEMENT_ID'].apply(classify_prefix)
    print(f"\n[{q}] ELEMENT_ID 유형 분포:")
    print(df_types.value_counts().to_string())


[2024_4Q] val.tsv 로딩 중...
  전체 로드: 5,604,149 rows, 메모리: 3.81 GB
  필터 결과: 177,474 rows (3.17%)
  고유 CIK: 2,350개
  고유 ELEMENT_ID: 3,928개

[2023_4Q] val.tsv 로딩 중...
  전체 로드: 3,828,791 rows, 메모리: 2.55 GB
  필터 결과: 84,302 rows (2.20%)
  고유 CIK: 2,245개
  고유 ELEMENT_ID: 2,877개

[2024_4Q] 필터 ELEMENT_ID 상위 30:
ELEMENT_ID
ifrs-full_OtherComprehensiveIncomeNetOfTaxGainsLossesOnRemeasurementsOfDefinedBenefitPlans                  40557
ifrs-full_LiabilityAssetOfDefinedBenefitPlans                                                                7771
dart_PostemploymentBenefitObligations                                                                        5050
ifrs-full_InterestExpenseIncomeNetDefinedBenefitLiabilityAsset                                               4627
ifrs-full_PaymentsFromPlanNetDefinedBenefitLiabilityAsset                                                    4603
dart_AdjustmentsForProvisionForSeveranceIndemnities                                                          4062
if

In [12]:
# =============================================================================
# Cell 3: lab.tsv 전체 로드 + 한글 레이블 필터 (Step 2-B)
# =============================================================================
import csv

# 한글 키워드 (LABEL 대상)
PENSION_KEYWORDS_KO = '|'.join([
    '확정급여', '퇴직급여', '퇴직연금', '사외적립', '퇴직연금운용',
    '할인율', '할인률', '임금상승', '급여상승', '보상증가',
    '듀레이션', '가중평균만기', '근무원가', '당기근무',
    '이자비용', '이자원가', '이자수익', '급여지급', '퇴직금지급',
    '민감도', '민감성', '보험수리', '재측정',
])

lab_filtered = {}
for q in QUARTERS:
    print(f"\n{'='*60}")
    print(f"[{q}] lab.tsv 로딩 중...")
    lpath = BASE / q / 'lab.tsv'
    
    df_lab = pd.read_csv(
        lpath, sep='\t',
        usecols=['CIK', 'REPORT_DATE', 'LABEL_ROLE_URI', 'ELMT_ID', 'TAXONOMY_ID', 'LANG', 'LABEL'],
        dtype=str,
        quoting=csv.QUOTE_NONE,      # LABEL에 따옴표 포함 → 파서 오류 방지
        on_bad_lines='warn',          # 파싱 불가 행은 경고 후 스킵
    )
    print(f"  전체: {len(df_lab):,} rows")
    
    # LANG='ko' 필터 (용량 절반 축소)
    df_lab = df_lab[df_lab['LANG'] == 'ko'].copy()
    print(f"  ko만: {len(df_lab):,} rows")
    
    # 한글 키워드 필터
    mask_ko = df_lab['LABEL'].str.contains(PENSION_KEYWORDS_KO, na=False)
    # 영문 키워드도 ELMT_ID에서 검색 (교집합 확보)
    mask_en = df_lab['ELMT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False)
    mask = mask_ko | mask_en
    
    lab_filtered[q] = df_lab[mask].copy()
    print(f"  필터 결과: {len(lab_filtered[q]):,} rows")
    print(f"  고유 ELMT_ID: {lab_filtered[q]['ELMT_ID'].nunique():,}개")
    
    del df_lab
    gc.collect()

# 레이블 역할별 분포 확인
for q in QUARTERS:
    print(f"\n[{q}] LABEL_ROLE_URI 분포:")
    print(lab_filtered[q]['LABEL_ROLE_URI'].value_counts().head(10).to_string())

# 표준 레이블(http://www.xbrl.org/2003/role/label)만 추출하여 간결한 매핑 생성
lab_std = {}
for q in QUARTERS:
    lf = lab_filtered[q]
    lab_std[q] = lf[lf['LABEL_ROLE_URI'] == 'http://www.xbrl.org/2003/role/label'][['CIK', 'ELMT_ID', 'LABEL']].drop_duplicates()
    print(f"\n[{q}] 표준 레이블 매핑: {len(lab_std[q]):,} rows, 고유 ELMT_ID: {lab_std[q]['ELMT_ID'].nunique():,}개")


[2024_4Q] lab.tsv 로딩 중...
  전체: 5,138,946 rows
  ko만: 2,563,424 rows
  필터 결과: 106,584 rows
  고유 ELMT_ID: 18,102개

[2023_4Q] lab.tsv 로딩 중...
  전체: 3,487,022 rows
  ko만: 1,737,523 rows
  필터 결과: 56,814 rows
  고유 ELMT_ID: 11,446개

[2024_4Q] LABEL_ROLE_URI 분포:
LABEL_ROLE_URI
http://www.xbrl.org/2003/role/label                      57690
http://www.xbrl.org/2003/role/terseLabel                 13790
http://dart.fss.or.kr/role/2024-06-30/ifrs/dart_label     8334
http://www.xbrl.org/2009/role/negatedLabel                6611
http://www.xbrl.org/2003/role/totalLabel                  5780
http://www.xbrl.org/2003/role/verboseLabel                3311
http://www.xbrl.org/2003/role/periodEndLabel              3000
http://www.xbrl.org/2003/role/periodStartLabel            2997
http://www.xbrl.org/2009/role/negatedTerseLabel           2126
http://www.xbrl.org/2003/role/commentaryGuidance          2083

[2023_4Q] LABEL_ROLE_URI 분포:
LABEL_ROLE_URI
http://www.xbrl.org/2003/role/label                  

In [13]:
# =============================================================================
# Cell 4: 변수별 ELEMENT_ID 후보 분류 (Step 2-C) — v2 정밀화
# =============================================================================
# 변경사항 v2:
#   1. en_pattern 정밀화 — OCI/재측정 혼입 방지 (negative lookahead)
#   2. UNIT_ID 기대값 추가 → Cell 5에서 필터링용
#   3. 민감도 변수 비활성 (커버리지 < 5 CIK)
#   4. exclude_pattern 추가 — 오매핑 방지

VARIABLES = {
    'DBO': {
        'desc': '확정급여채무 (총액)',
        'en_pattern': r'DefinedBenefitObligation(?:AtPresentValue)?$|PresentValueOfDefinedBenefitObligation',
        'exclude_pattern': r'OtherComprehensiveIncome|Remeasurement|GainLoss|Increase.*Decrease|Sensitivity|ChangeIn',
        'ko_keywords': ['확정급여채무', '확정급여부채', '퇴직급여채무', '퇴직급여부채'],
        'expected_unit': 'KRW',
    },
    'PlanAsset': {
        'desc': '사외적립자산 공정가치',
        'en_pattern': r'PlanAssetsAtFairValue|FairValueOfPlanAssets',
        'exclude_pattern': r'OtherComprehensiveIncome|Remeasurement|GainLoss|Increase.*Decrease|AdjustmentsFor',
        'ko_keywords': ['사외적립자산', '퇴직연금운용자산'],
        'expected_unit': 'KRW',
    },
    'NetDBO': {
        'desc': '순확정급여부채(자산)',
        'en_pattern': r'NetDefinedBenefitLiability|RecognisedLiabilit.*DefinedBenefit|LiabilityAssetOfDefinedBenefit|PostemploymentBenefitObligations',
        'exclude_pattern': r'OtherComprehensiveIncome|Remeasurement|GainLoss|Increase.*Decrease|CurrentServiceCost|InterestExpense|InterestIncome|Payment|Sensitivity',
        'ko_keywords': ['순확정급여부채', '순확정급여자산', '순확정급여채무'],
        'expected_unit': 'KRW',
    },
    'DiscountRate': {
        'desc': '할인율',
        'en_pattern': r'ActuarialAssumptionOfDiscountRate|DiscountRateForDefinedBenefit',
        'exclude_pattern': r'Sensitivity|IncreaseDecrease|OtherProvisions',
        'ko_keywords': ['할인율', '할인률'],
        'expected_unit': 'PURE',
    },
    'SalaryGrowth': {
        'desc': '임금상승률',
        'en_pattern': r'ExpectedRatesOfSalaryIncreases|ActuarialAssumption.*Salary|RateOfCompensationIncrease',
        'exclude_pattern': r'Sensitivity|IncreaseDecrease',
        'ko_keywords': ['임금상승률', '급여상승률', '보상증가율'],
        'expected_unit': 'PURE',
    },
    'Duration': {
        'desc': 'DBO 듀레이션 (가중평균만기)',
        'en_pattern': r'WeightedAverageDuration.*DefinedBenefit',
        'exclude_pattern': None,
        'ko_keywords': ['듀레이션', '가중평균만기'],
        'expected_unit': 'PURE',
    },
    'ServiceCost': {
        'desc': '당기근무원가',
        'en_pattern': r'CurrentServiceCost.*DefinedBenefit|CurrentServiceCostNetDefinedBenefit',
        'exclude_pattern': None,
        'ko_keywords': ['당기근무원가'],
        'expected_unit': 'KRW',
    },
    'InterestCost': {
        'desc': '이자비용 (DBO 이자원가)',
        'en_pattern': r'InterestExpense.*DefinedBenefit|InterestCost.*DefinedBenefit',
        'exclude_pattern': r'AdjustmentsFor(?!.*DefinedBenefit)',
        'ko_keywords': ['이자비용.*확정급여', '이자원가.*확정급여'],
        'expected_unit': 'KRW',
    },
    'InterestIncome': {
        'desc': '이자수익 (사외적립자산)',
        'en_pattern': r'InterestIncome.*DefinedBenefit|InterestIncome.*PlanAsset|ExpectedReturnOnPlanAsset',
        'exclude_pattern': r'AdjustmentsFor(?!.*DefinedBenefit)',
        'ko_keywords': ['이자수익.*사외', '이자수익.*적립'],
        'expected_unit': 'KRW',
    },
    'BenefitPayment': {
        'desc': '급여지급액',
        'en_pattern': r'PaymentsFromPlan.*DefinedBenefit|BenefitsPaid.*DefinedBenefit',
        'exclude_pattern': None,
        'ko_keywords': ['급여지급', '퇴직금지급'],
        'expected_unit': 'KRW',
    },
    'ActuarialGL': {
        'desc': '보험수리적손익 (OCI)',
        'en_pattern': r'OtherComprehensiveIncome.*Remeasurement.*DefinedBenefit|ActuarialGainsLosses.*DefinedBenefit|GainsLossesOnRemeasurement.*DefinedBenefit',
        'exclude_pattern': None,
        'ko_keywords': ['보험수리.*손익', '재측정.*확정급여'],
        'expected_unit': 'KRW',
    },
    'RetirementBenefitCost': {
        'desc': '퇴직급여(비용 합계)',
        'en_pattern': r'RetirementBenefitExpense|PostemploymentBenefitExpense',
        'exclude_pattern': r'Obligations|Liability|Asset|PlanAsset',
        'ko_keywords': ['퇴직급여.*비용'],
        'expected_unit': 'KRW',
    },
    'NetInterest': {
        'desc': '순이자(순금융원가)',
        'en_pattern': r'NetInterest.*DefinedBenefit',
        'exclude_pattern': None,
        'ko_keywords': ['순이자.*확정급여', '순금융원가'],
        'expected_unit': 'KRW',
    },
}

# 민감도 변수 제외 (커버리지 < 5 CIK — 수집 불가)
# 'SensitivityDiscount', 'SensitivitySalary' 삭제

print(f"분석 대상 변수: {len(VARIABLES)}개 (민감도 2개 제외)")
for k, v in VARIABLES.items():
    print(f"  {k:25s} → {v['desc']} (UNIT: {v['expected_unit']})")

# --- 사전 집계: 분기별 ELEMENT_ID→(n_cik, n_rows, dominant_unit) ---
eid_stats = {}
for q in QUARTERS:
    vf = val_filtered[q]
    stats = vf.groupby('ELEMENT_ID').agg(
        n_cik=('CIK', 'nunique'),
        n_rows=('CIK', 'size'),
        dominant_unit=('UNIT_ID', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else ''),
    )
    eid_stats[q] = stats

# --- 사전 집계: lab ELMT_ID → 최빈 한글 레이블 ---
lab_eid_label = {}
for q in QUARTERS:
    ls = lab_std[q]
    lab_eid_label[q] = ls.groupby('ELMT_ID')['LABEL'].agg(lambda x: x.value_counts().index[0]).to_dict()

# --- 변수별 ELEMENT_ID 후보 추출 ---
var_candidates = {}

for var_name, var_def in VARIABLES.items():
    print(f"\n{'='*70}")
    print(f"변수: {var_name} ({var_def['desc']}) [기대 UNIT: {var_def['expected_unit']}]")
    print(f"{'='*70}")
    
    results = []
    for q in QUARTERS:
        vf = val_filtered[q]
        ls = lab_std[q]
        stats = eid_stats[q]
        lbl = lab_eid_label[q]
        
        # val에서 영문 패턴 매칭
        mask_val = vf['ELEMENT_ID'].str.contains(var_def['en_pattern'], case=False, na=False)
        # exclude 패턴 적용
        if var_def.get('exclude_pattern'):
            mask_excl = vf['ELEMENT_ID'].str.contains(var_def['exclude_pattern'], case=False, na=False)
            mask_val = mask_val & ~mask_excl
        matched_eids_val = set(vf[mask_val]['ELEMENT_ID'].unique())
        
        # lab에서 한글 키워드 매칭 → ELMT_ID 추출
        ko_pattern = '|'.join(var_def['ko_keywords'])
        mask_lab = ls['LABEL'].str.contains(ko_pattern, na=False)
        matched_eids_lab = set(ls[mask_lab]['ELMT_ID'].unique())
        
        # lab 결과에도 exclude 적용
        if var_def.get('exclude_pattern'):
            matched_eids_lab = {eid for eid in matched_eids_lab
                                if not re.search(var_def['exclude_pattern'], eid, re.IGNORECASE)}
        
        all_eids = matched_eids_val | matched_eids_lab
        
        for eid in sorted(all_eids):
            if eid in stats.index:
                n_cik = int(stats.loc[eid, 'n_cik'])
                n_rows = int(stats.loc[eid, 'n_rows'])
                dom_unit = stats.loc[eid, 'dominant_unit']
            else:
                n_cik, n_rows, dom_unit = 0, 0, ''
            
            # UNIT_ID 불일치 시 제외 (entity 태그에서 오매핑 방지)
            expected = var_def['expected_unit']
            if dom_unit and expected and dom_unit != expected:
                # 단, Duration은 NaN 허용 (UNIT_ID 미기재 빈번)
                if var_name == 'Duration' and (dom_unit == '' or pd.isna(dom_unit)):
                    pass
                else:
                    continue  # UNIT 불일치 → 이 ELEMENT_ID 스킵
            
            if eid.startswith('ifrs-full_'):
                eid_type = 'ifrs-full'
            elif eid.startswith('dart_'):
                eid_type = 'dart'
            elif eid.startswith('entity'):
                eid_type = 'entity'
            else:
                eid_type = 'other'
            
            ko_label = lbl.get(eid, '')
            
            source = []
            if eid in matched_eids_val:
                source.append('val_EN')
            if eid in matched_eids_lab:
                source.append('lab_KO')
            
            results.append({
                'variable': var_name, 'quarter': q,
                'ELEMENT_ID': eid, 'type': eid_type,
                'n_cik': n_cik, 'n_rows': n_rows,
                'dominant_unit': dom_unit,
                'ko_label': ko_label,
                'match_source': '+'.join(source),
            })
    
    df_cand = pd.DataFrame(results)
    var_candidates[var_name] = df_cand
    
    if len(df_cand) == 0:
        print("  매칭 결과 없음!")
        continue
    
    # 표준/DART 레벨 요약 (entity 제외)
    non_entity = df_cand[df_cand['type'] != 'entity']
    if len(non_entity) > 0:
        print("\n  [표준/DART ELEMENT_ID]:")
        summary = non_entity.groupby(['ELEMENT_ID', 'type', 'ko_label', 'dominant_unit']).agg(
            quarters=('quarter', lambda x: ', '.join(sorted(x.unique()))),
            max_cik=('n_cik', 'max'),
            total_rows=('n_rows', 'sum'),
        ).reset_index().sort_values('max_cik', ascending=False)
        for _, row in summary.iterrows():
            print(f"    {row['ELEMENT_ID']:65s} [{row['type']:10s}] CIK={row['max_cik']:>5,} UNIT={row['dominant_unit']:4s} | {row['ko_label']}")
    
    n_entity = df_cand[df_cand['type'] == 'entity']['ELEMENT_ID'].nunique()
    if n_entity > 0:
        print(f"\n  [기업확장 ELEMENT_ID]: {n_entity}개 (고유)")


분석 대상 변수: 13개 (민감도 2개 제외)
  DBO                       → 확정급여채무 (총액) (UNIT: KRW)
  PlanAsset                 → 사외적립자산 공정가치 (UNIT: KRW)
  NetDBO                    → 순확정급여부채(자산) (UNIT: KRW)
  DiscountRate              → 할인율 (UNIT: PURE)
  SalaryGrowth              → 임금상승률 (UNIT: PURE)
  Duration                  → DBO 듀레이션 (가중평균만기) (UNIT: PURE)
  ServiceCost               → 당기근무원가 (UNIT: KRW)
  InterestCost              → 이자비용 (DBO 이자원가) (UNIT: KRW)
  InterestIncome            → 이자수익 (사외적립자산) (UNIT: KRW)
  BenefitPayment            → 급여지급액 (UNIT: KRW)
  ActuarialGL               → 보험수리적손익 (OCI) (UNIT: KRW)
  RetirementBenefitCost     → 퇴직급여(비용 합계) (UNIT: KRW)
  NetInterest               → 순이자(순금융원가) (UNIT: KRW)

변수: DBO (확정급여채무 (총액)) [기대 UNIT: KRW]

  [표준/DART ELEMENT_ID]:
    dart_PostemploymentBenefitObligations                             [dart      ] CIK=1,401 UNIT=KRW  | 퇴직급여부채
    ifrs-full_LiabilityAssetOfDefinedBenefitPlans                     [ifrs-full ] CIK=  456 UNIT=KRW  | 순

In [14]:
# =============================================================================
# Cell 4.5: CIK 00101044 강화 진단 — ConsolidatedMember 엄격 필터 + 3개년 × 13변수
# =============================================================================
# Cell 4의 VARIABLES, var_candidates, eid_stats, lab_eid_label, lab_std 재사용
# 최신 보고서 우선 원칙: 이후 보고서의 비교재무제표가 재작성 반영하므로 더 정확
#
# v2 수정사항 (2026-02-25):
#   1. Dimension축 처리: PlanAssetsMember penalty, DBO-ObligationMember bonus
#   2. DiscountRate: ActuarialAssumption bonus, CashFlowProjection penalty
#   3. BenefitPayment: DBO-side(ObligationMember) 선호 (부채인덱스 용도)

TARGET_CIK = '00101044'

# ─────────────────────────────────────────────────────────────────────────────
# [A] 기간 소스 정의 + ConsolidatedMember 엄격 필터
# ─────────────────────────────────────────────────────────────────────────────
PERIOD_SOURCES = [
    # (표시명, quarter_key, period_prefix, 설명)
    ('2024.12', '2024_4Q', 'CFY', '2024_4Q 당기'),
    ('2023.12', '2024_4Q', 'PFY', '2024_4Q 전기 비교재무'),
    ('2022.12', '2023_4Q', 'PFY', '2023_4Q 전기 비교재무'),
]

# BS vs PL 기간 유형
VAR_PERIOD_TYPE = {
    'DBO': 'eFY', 'PlanAsset': 'eFY', 'NetDBO': 'eFY',
    'DiscountRate': 'eFY', 'SalaryGrowth': 'eFY', 'Duration': 'eFY',
    'ServiceCost': 'dFY', 'InterestCost': 'dFY', 'InterestIncome': 'dFY',
    'BenefitPayment': 'dFY', 'ActuarialGL': 'dFY',
    'RetirementBenefitCost': 'dFY', 'NetInterest': 'dFY',
}

# ─── Dimension축 + EID 보정 규칙 ──────────────────────────────────────────────
# Dimension axis 멤버: CONTEXT_ID 내 포함 여부로 판단
# - PlanAssetsMember가 포함된 행 = 사외적립자산 측 데이터
# - PresentValueOfDefinedBenefitObligationMember = DBO(의무) 측 데이터
# 부채인덱스 산출 목적이므로 DBO 측 선호

# 변수별 Dimension 선호도 (부채인덱스 관점)
# 'dbo_side': ObligationMember 선호, PlanAssetsMember 회피
# 'pa_side':  PlanAssetsMember 선호
# 'neutral':  Dimension 무관
VAR_DIM_PREFERENCE = {
    'DBO': 'neutral',           # DBO 자체는 보통 Dimension 없음
    'PlanAsset': 'pa_side',     # PA는 PA 측이 맞음
    'NetDBO': 'neutral',        # 순액은 Dimension 없는게 일반적
    'DiscountRate': 'dbo_side', # 할인율은 DBO 측 가정
    'SalaryGrowth': 'neutral',
    'Duration': 'dbo_side',
    'ServiceCost': 'dbo_side',
    'InterestCost': 'dbo_side',
    'InterestIncome': 'pa_side',   # 이자수익은 PA 측
    'BenefitPayment': 'dbo_side',  # 부채인덱스: DBO 측 지급액
    'ActuarialGL': 'dbo_side',
    'RetirementBenefitCost': 'neutral',
    'NetInterest': 'neutral',
}

# DiscountRate 전용 EID 보정 패턴
# ActuarialAssumption = 퇴직연금 보험수리 가정 (정확)
# CashFlowProjection / GoodwillImpairment = 사업가치 할인율 (오매핑)
DISCOUNT_RATE_BONUS_PATTERNS = ['ActuarialAssumption']
DISCOUNT_RATE_PENALTY_PATTERNS = ['CashFlowProjection', 'GoodwillImpairment',
                                   'RiskFreeInterestRate']

print(f"{'='*80}")
print(f"CIK {TARGET_CIK} 강화 진단 — ConsolidatedMember 엄격 필터 + 3개년 × 13변수 (v2)")
print(f"{'='*80}")

# 기간 소스별 데이터 가용 여부 확인
available_quarters = {}
for label, qkey, prefix, desc in PERIOD_SOURCES:
    if qkey in val_filtered:
        has_cik = (val_filtered[qkey]['CIK'] == TARGET_CIK).any()
        available_quarters[(label, qkey, prefix)] = has_cik
        status = '✓ 데이터 존재' if has_cik else '✗ CIK 미존재'
        print(f"  {label} ({desc}): {status}")
    else:
        available_quarters[(label, qkey, prefix)] = False
        print(f"  {label} ({desc}): ✗ quarter 미로드")

# ─────────────────────────────────────────────────────────────────────────────
# [B] 13개 변수 × 3개년 매핑 (best-row 선택)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print("[B] 변수별 매핑 결과 (v2: Dimension축 + EID 보정 적용)")
print(f"{'─'*80}")

results = []
anomaly_details = []  # [E]용 이상 항목 수집

for var_name, var_def in VARIABLES.items():
    expected_unit = var_def['expected_unit']
    period_type = VAR_PERIOD_TYPE[var_name]
    dim_pref = VAR_DIM_PREFERENCE.get(var_name, 'neutral')

    for period_label, qkey, period_prefix, period_desc in PERIOD_SOURCES:
        # quarter 가용성 체크
        if not available_quarters.get((period_label, qkey, period_prefix), False):
            results.append({
                'variable': var_name, 'period': period_label,
                'quarter_src': qkey, 'prefix': period_prefix,
                'ELEMENT_ID': None, 'ko_label': None,
                'VALUE': None, 'VALUE_억': None,
                'UNIT_ID': None, 'CONTEXT_ID': None,
                'eid_type': None, 'score': None,
                'status': '데이터없음', 'note': f'{qkey}에 CIK 미존재',
            })
            continue

        vf = val_filtered[qkey]
        cik_data = vf[vf['CIK'] == TARGET_CIK].copy()

        # var_candidates에서 해당 변수의 ELEMENT_ID 후보 목록
        if var_name not in var_candidates or len(var_candidates[var_name]) == 0:
            results.append({
                'variable': var_name, 'period': period_label,
                'quarter_src': qkey, 'prefix': period_prefix,
                'ELEMENT_ID': None, 'ko_label': None,
                'VALUE': None, 'VALUE_억': None,
                'UNIT_ID': None, 'CONTEXT_ID': None,
                'eid_type': None, 'score': None,
                'status': '후보없음', 'note': '전역 ELEMENT_ID 후보 없음',
            })
            continue

        cand_eids = set(
            var_candidates[var_name][var_candidates[var_name]['quarter'] == qkey]['ELEMENT_ID']
        )
        if not cand_eids:
            results.append({
                'variable': var_name, 'period': period_label,
                'quarter_src': qkey, 'prefix': period_prefix,
                'ELEMENT_ID': None, 'ko_label': None,
                'VALUE': None, 'VALUE_억': None,
                'UNIT_ID': None, 'CONTEXT_ID': None,
                'eid_type': None, 'score': None,
                'status': '후보없음', 'note': f'{qkey} ELEMENT_ID 후보 없음',
            })
            continue

        # 1) ELEMENT_ID 후보 필터
        subset = cik_data[cik_data['ELEMENT_ID'].isin(cand_eids)].copy()

        # 2) ConsolidatedMember 엄격 필터 (SeparateMember 무조건 제외)
        subset = subset[~subset['CONTEXT_ID'].str.contains('SeparateMember', na=False)]
        con_mask = subset['CONTEXT_ID'].str.contains('ConsolidatedMember', na=False)
        if con_mask.any():
            subset = subset[con_mask]

        # 3) 기간 prefix 필터 (CFY / PFY)
        subset = subset[subset['CONTEXT_ID'].str.contains(period_prefix, na=False)]

        # 4) BS(eFY) / PL(dFY) 필터
        subset = subset[subset['CONTEXT_ID'].str.contains(period_type, na=False)]

        # 5) UNIT_ID 필터
        unit_mask = (subset['UNIT_ID'] == expected_unit) | subset['UNIT_ID'].isna()
        subset = subset[unit_mask]

        if len(subset) == 0:
            results.append({
                'variable': var_name, 'period': period_label,
                'quarter_src': qkey, 'prefix': period_prefix,
                'ELEMENT_ID': None, 'ko_label': None,
                'VALUE': None, 'VALUE_억': None,
                'UNIT_ID': None, 'CONTEXT_ID': None,
                'eid_type': None, 'score': None,
                'status': '매핑없음', 'note': 'ConsolidatedMember+기간+UNIT 필터 후 0건',
            })
            continue

        # 6) 스코어 계산 (v2: 3단계 보정)
        #    [기본] ELEMENT_ID prefix: ifrs-full(+8) > dart(+4) > entity(+0)
        #    [보정1] Dimension축: DBO-side/PA-side 선호도 반영
        #    [보정2] EID 패턴: DiscountRate의 ActuarialAssumption vs CashFlowProjection
        #    [보정3] DBO 변수: '순' 레이블 penalty (기존)

        def _base_score(eid):
            """ELEMENT_ID prefix 우선순위"""
            if eid.startswith('ifrs-full_'):
                return 8
            elif eid.startswith('dart_'):
                return 4
            return 0

        subset['_score'] = subset['ELEMENT_ID'].apply(_base_score)

        # [보정1] Dimension축 보정
        if dim_pref == 'dbo_side':
            # DBO/Obligation 측 선호: ObligationMember 보너스, PlanAssetsMember 페널티
            pa_member = subset['CONTEXT_ID'].str.contains('PlanAssetsMember', na=False)
            obl_member = subset['CONTEXT_ID'].str.contains(
                'PresentValueOfDefinedBenefitObligationMember|ObligationMember', 
                na=False, regex=True
            )
            subset.loc[pa_member, '_score'] -= 15
            subset.loc[obl_member, '_score'] += 5
        elif dim_pref == 'pa_side':
            # PA 측 선호: PlanAssetsMember 보너스, ObligationMember 페널티
            pa_member = subset['CONTEXT_ID'].str.contains('PlanAssetsMember', na=False)
            obl_member = subset['CONTEXT_ID'].str.contains(
                'PresentValueOfDefinedBenefitObligationMember|ObligationMember',
                na=False, regex=True
            )
            subset.loc[pa_member, '_score'] += 5
            subset.loc[obl_member, '_score'] -= 15

        # [보정2] DiscountRate 전용 EID 패턴 보정
        if var_name == 'DiscountRate':
            for pattern in DISCOUNT_RATE_BONUS_PATTERNS:
                bonus_mask = subset['ELEMENT_ID'].str.contains(pattern, na=False)
                subset.loc[bonus_mask, '_score'] += 5
            for pattern in DISCOUNT_RATE_PENALTY_PATTERNS:
                penalty_mask = subset['ELEMENT_ID'].str.contains(pattern, na=False)
                subset.loc[penalty_mask, '_score'] -= 15

        # [보정3] DBO 변수: '순' 레이블 penalty (-10)
        if var_name == 'DBO':
            lkup = lab_lookup.get(qkey, {}) if 'lab_lookup' in dir() else {}
            if not lkup and qkey in lab_std:
                lkup = lab_std[qkey].drop_duplicates(
                    subset=['CIK', 'ELMT_ID']
                ).set_index(['CIK', 'ELMT_ID'])['LABEL'].to_dict()
            labels_s = subset.apply(
                lambda r: lkup.get((r['CIK'], r['ELEMENT_ID']), ''), axis=1
            )
            net_penalty = labels_s.str.contains('순확정|순 확정', na=False).astype(int) * (-10)
            subset['_score'] = subset['_score'] + net_penalty

        # 7) 최고 스코어 행 선택
        best_idx = subset['_score'].idxmax()
        best = subset.loc[best_idx]

        # 한글 레이블 조회
        ko_label = ''
        if qkey in lab_eid_label:
            ko_label = lab_eid_label[qkey].get(best['ELEMENT_ID'], '')
        if not ko_label and qkey in lab_std:
            cik_lab = lab_std[qkey][
                (lab_std[qkey]['CIK'] == TARGET_CIK) &
                (lab_std[qkey]['ELMT_ID'] == best['ELEMENT_ID'])
            ]
            if len(cik_lab) > 0:
                ko_label = cik_lab.iloc[0]['LABEL']

        # VALUE 파싱
        val_raw = best['VALUE']
        try:
            val_num = float(val_raw)
        except (ValueError, TypeError):
            val_num = None
        val_억 = val_num / 1e8 if val_num is not None else None

        eid_type = ('ifrs-full' if best['ELEMENT_ID'].startswith('ifrs-full_')
                    else 'dart' if best['ELEMENT_ID'].startswith('dart_')
                    else 'entity')

        # 이상 판정
        status = '정상'
        note_parts = []
        n_candidates = len(subset)

        if eid_type == 'entity':
            status = '주의'
            note_parts.append('entity 태그 사용')
        if n_candidates > 1:
            status = '주의' if status == '정상' else status
            note_parts.append(f'복수 후보 {n_candidates}건')
        if val_num is not None and val_num == 0:
            status = '검토'
            note_parts.append('값 0')
        if val_num is not None and val_num < 0:
            status = '검토'
            note_parts.append('음수값')
        if var_name == 'DBO' and ko_label and '순' in ko_label:
            status = '검토'
            note_parts.append('DBO/NetDBO 혼동 가능')

        # Dimension축 선택 결과 표시
        ctx_str = str(best['CONTEXT_ID'])
        if 'PlanAssetsMember' in ctx_str and dim_pref == 'dbo_side':
            status = '검토'
            note_parts.append('PA측 Dimension 선택됨(DBO측 기대)')
        elif 'ObligationMember' in ctx_str and dim_pref == 'pa_side':
            status = '검토'
            note_parts.append('DBO측 Dimension 선택됨(PA측 기대)')

        note = '; '.join(note_parts) if note_parts else ''

        results.append({
            'variable': var_name, 'period': period_label,
            'quarter_src': qkey, 'prefix': period_prefix,
            'ELEMENT_ID': best['ELEMENT_ID'], 'ko_label': ko_label,
            'VALUE': val_raw, 'VALUE_억': val_억,
            'UNIT_ID': best['UNIT_ID'], 'CONTEXT_ID': best['CONTEXT_ID'],
            'eid_type': eid_type, 'score': best['_score'],
            'status': status, 'note': note,
        })

        # 이상 항목이면 상세 수집
        if status in ('주의', '검토'):
            all_rows = cik_data[cik_data['ELEMENT_ID'].isin(cand_eids)].copy()
            anomaly_details.append({
                'variable': var_name, 'period': period_label,
                'status': status, 'note': note,
                'selected_eid': best['ELEMENT_ID'],
                'selected_value': val_raw,
                'all_candidates': all_rows[
                    ['ELEMENT_ID', 'CONTEXT_ID', 'UNIT_ID', 'VALUE']
                ].to_dict('records'),
            })

df_results = pd.DataFrame(results)

# ─────────────────────────────────────────────────────────────────────────────
# [C] 요약 매트릭스 출력
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print("[C] 요약 매트릭스 — 13변수 × 3개년 (v2)")
print(f"{'─'*80}")

def format_value(row):
    if row['ELEMENT_ID'] is None:
        return '-'
    if row['UNIT_ID'] == 'KRW' and row['VALUE_억'] is not None:
        return f"{row['VALUE_억']:,.1f}억"
    elif row['VALUE'] is not None:
        try:
            v = float(row['VALUE'])
            if abs(v) < 1:
                return f"{v*100:.2f}%"
            return f"{v:,.2f}"
        except (ValueError, TypeError):
            return str(row['VALUE'])
    return '-'

df_results['display_val'] = df_results.apply(format_value, axis=1)

VAR_ORDER = ['DBO', 'PlanAsset', 'NetDBO', 'DiscountRate', 'SalaryGrowth',
             'Duration', 'ServiceCost', 'InterestCost', 'InterestIncome',
             'BenefitPayment', 'ActuarialGL', 'RetirementBenefitCost', 'NetInterest']

pivot_val = df_results.pivot_table(
    index='variable', columns='period', values='display_val', aggfunc='first'
)
pivot_status = df_results.pivot_table(
    index='variable', columns='period', values='status', aggfunc='first'
)

eid_info = df_results[df_results['period'] == '2024.12'].set_index('variable')[
    ['ELEMENT_ID', 'ko_label', 'eid_type']
]

period_cols = ['2024.12', '2023.12', '2022.12']
existing_cols = [c for c in period_cols if c in pivot_val.columns]
pivot_val = pivot_val.reindex(index=VAR_ORDER, columns=existing_cols)
pivot_status = pivot_status.reindex(index=VAR_ORDER, columns=existing_cols)
eid_info = eid_info.reindex(index=VAR_ORDER)

summary = pd.DataFrame(index=VAR_ORDER)
summary['BS/PL'] = [VAR_PERIOD_TYPE[v] for v in VAR_ORDER]
summary['ELEMENT_ID'] = eid_info['ELEMENT_ID'].fillna('-')
summary['한글레이블'] = eid_info['ko_label'].fillna('-')
for col in existing_cols:
    summary[col] = pivot_val[col].fillna('-')
    summary[f'{col}_상태'] = pivot_status[col].fillna('-')

REQUIRED = {'DBO', 'PlanAsset', 'DiscountRate', 'SalaryGrowth', 'Duration',
            'ServiceCost', 'InterestCost'}
summary.insert(0, '구분', ['★필수' if v in REQUIRED else '○선택' for v in VAR_ORDER])

print(f"\nCIK: {TARGET_CIK}")
print(f"기간 소스: 2024.12(2024_4Q CFY) | 2023.12(2024_4Q PFY) | 2022.12(2023_4Q PFY)")
print()
display(summary)

for col in existing_cols:
    status_col = f'{col}_상태'
    if status_col in summary.columns:
        status_counts = summary[status_col].value_counts()
        print(f"\n{col} 상태: {dict(status_counts)}")

# ─────────────────────────────────────────────────────────────────────────────
# [D] DBO 총액/순액 판별 (단일 CIK 버전)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print("[D] DBO 총액/순액 판별")
print(f"{'─'*80}")

q_diag = '2024_4Q'
vf_diag = val_filtered[q_diag]
cik_data_diag = vf_diag[vf_diag['CIK'] == TARGET_CIK]

ifrs_dbo_eids_diag = [
    'ifrs-full_DefinedBenefitObligationAtPresentValue',
    'ifrs-full_PresentValueOfDefinedBenefitObligation',
]
ifrs_pa_eids_diag = [
    'ifrs-full_PlanAssetsAtFairValue',
    'ifrs-full_FairValueOfPlanAssets',
]

def get_best_consolidated_value(data, target_eids, period_pref='CFY', pt='eFY'):
    subset = data[data['ELEMENT_ID'].isin(target_eids)].copy()
    subset = subset[~subset['CONTEXT_ID'].str.contains('SeparateMember', na=False)]
    con = subset[subset['CONTEXT_ID'].str.contains('ConsolidatedMember', na=False)]
    if len(con) > 0:
        subset = con
    subset = subset[subset['CONTEXT_ID'].str.contains(period_pref, na=False)]
    subset = subset[subset['CONTEXT_ID'].str.contains(pt, na=False)]
    # Dimension축 필터: PlanAssetsMember/ObligationMember 없는 행 우선 (총계)
    no_dim = subset[~subset['CONTEXT_ID'].str.contains(
        'PlanAssetsMember|ObligationMember', na=False, regex=True)]
    if len(no_dim) > 0:
        subset = no_dim
    if len(subset) == 0:
        return None, None
    row = subset.iloc[0]
    try:
        return float(row['VALUE']), row['ELEMENT_ID']
    except (ValueError, TypeError):
        return None, row['ELEMENT_ID']

ifrs_dbo_val, ifrs_dbo_eid = get_best_consolidated_value(cik_data_diag, ifrs_dbo_eids_diag)
pa_val, pa_eid = get_best_consolidated_value(cik_data_diag, ifrs_pa_eids_diag)

dbo_result = df_results[(df_results['variable'] == 'DBO') & (df_results['period'] == '2024.12')]
mapped_dbo_val = None
mapped_dbo_eid = None
if len(dbo_result) > 0 and dbo_result.iloc[0]['VALUE'] is not None:
    try:
        mapped_dbo_val = float(dbo_result.iloc[0]['VALUE'])
        mapped_dbo_eid = dbo_result.iloc[0]['ELEMENT_ID']
    except (ValueError, TypeError):
        pass

net_result = df_results[(df_results['variable'] == 'NetDBO') & (df_results['period'] == '2024.12')]
mapped_net_val = None
if len(net_result) > 0 and net_result.iloc[0]['VALUE'] is not None:
    try:
        mapped_net_val = float(net_result.iloc[0]['VALUE'])
    except (ValueError, TypeError):
        pass

print(f"\n  IFRS DBO (총액): {ifrs_dbo_val:,.0f} ({ifrs_dbo_eid})" if ifrs_dbo_val else "  IFRS DBO (총액): 없음")
print(f"  사외적립자산 (PA): {pa_val:,.0f} ({pa_eid})" if pa_val else "  사외적립자산 (PA): 없음")
print(f"  매핑 DBO 값: {mapped_dbo_val:,.0f} ({mapped_dbo_eid})" if mapped_dbo_val else "  매핑 DBO 값: 없음")
print(f"  매핑 NetDBO 값: {mapped_net_val:,.0f}" if mapped_net_val else "  매핑 NetDBO 값: 없음")

dbo_type = '판별불가'
판별근거 = []

if ifrs_dbo_val is not None and pa_val is not None:
    판별근거.append(f'IFRS DBO({ifrs_dbo_val:,.0f}) + PA({pa_val:,.0f}) 별도 존재')
    net_calc = ifrs_dbo_val - pa_val
    판별근거.append(f'계산 Net = DBO - PA = {net_calc:,.0f}')

    if mapped_dbo_val is not None:
        ratio = mapped_dbo_val / net_calc if net_calc != 0 else None
        if ratio and 0.95 <= abs(ratio) <= 1.05:
            dbo_type = '순액(DBO매핑=Net)'
            판별근거.append(f'매핑DBO/계산Net = {ratio:.4f} → DBO매핑값이 실제 순액')
        elif ifrs_dbo_val != 0 and abs(mapped_dbo_val - ifrs_dbo_val) / abs(ifrs_dbo_val) < 0.05:
            dbo_type = '총액(DBO매핑≈IFRS_DBO)'
            판별근거.append(f'매핑DBO ≈ IFRS_DBO (차이 {abs(mapped_dbo_val-ifrs_dbo_val)/abs(ifrs_dbo_val)*100:.2f}%)')
        else:
            dbo_type = '총액분리'
            판별근거.append('DBO매핑값 ≠ Net, ≠ IFRS_DBO → 별도 확인 필요')
    else:
        dbo_type = '총액분리'
elif ifrs_dbo_val is not None and pa_val is None:
    dbo_type = '총액(PA없음)'
    판별근거.append('IFRS DBO 존재, PA 없음')
else:
    if len(dbo_result) > 0:
        label = str(dbo_result.iloc[0].get('ko_label', ''))
        if '순확정급여' in label:
            dbo_type = '순액(레이블)'
            판별근거.append(f'레이블: "{label}"')
        elif '확정급여채무' in label:
            dbo_type = '총액추정(레이블)'
            판별근거.append(f'레이블: "{label}"')
        elif '퇴직급여부채' in label:
            dbo_type = '순액추정(퇴직급여부채)'
            판별근거.append(f'레이블: "{label}"')
        else:
            판별근거.append(f'레이블: "{label}" → 판별 어려움')

print(f"\n  ▶ DBO 유형: {dbo_type}")
for g in 판별근거:
    print(f"    - {g}")

# ─────────────────────────────────────────────────────────────────────────────
# [E] 이상 항목 상세 (수동 검토 대상)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print("[E] 이상 항목 상세 — 수동 검토 대상 (v2)")
print(f"{'─'*80}")

status_all = df_results['status'].value_counts()
print(f"\n전체 상태 집계: {dict(status_all)}")

검토_items = df_results[df_results['status'] == '검토']
주의_items = df_results[df_results['status'] == '주의']
매핑없음_items = df_results[df_results['status'].isin(['매핑없음', '후보없음', '데이터없음'])]

if len(검토_items) > 0:
    print(f"\n── 🔴 검토 필요 ({len(검토_items)}건) ──")
    for _, row in 검토_items.iterrows():
        print(f"\n  [{row['variable']}] {row['period']} — {row['note']}")
        print(f"    선택: {row['ELEMENT_ID']} (score={row['score']})")
        print(f"    값: {row['VALUE']} | UNIT: {row['UNIT_ID']} | CTX: {row['CONTEXT_ID']}")
        for ad in anomaly_details:
            if ad['variable'] == row['variable'] and ad['period'] == row['period']:
                print(f"    ── 전체 후보 ({len(ad['all_candidates'])}건) ──")
                for c in ad['all_candidates']:
                    marker = ' ◀선택' if c['ELEMENT_ID'] == ad['selected_eid'] else ''
                    print(f"      EID: {c['ELEMENT_ID']}")
                    print(f"        CTX: {c['CONTEXT_ID']} | UNIT: {c['UNIT_ID']} | VAL: {c['VALUE']}{marker}")

if len(주의_items) > 0:
    print(f"\n── 🟡 주의 ({len(주의_items)}건) ──")
    for _, row in 주의_items.iterrows():
        print(f"  [{row['variable']}] {row['period']} — {row['note']}")
        print(f"    선택: {row['ELEMENT_ID']} (score={row['score']}) = {row['VALUE']} ({row['ko_label']})")

if len(매핑없음_items) > 0:
    print(f"\n── ⚪ 매핑 없음 ({len(매핑없음_items)}건) ──")
    for _, row in 매핑없음_items.iterrows():
        print(f"  [{row['variable']}] {row['period']} — {row['status']}: {row['note']}")

# ConsolidatedMember 필터 검증
print(f"\n── ConsolidatedMember 필터 검증 ──")
sep_count = 0
for _, row in df_results.iterrows():
    ctx = row.get('CONTEXT_ID')
    if ctx and 'SeparateMember' in str(ctx):
        sep_count += 1
        print(f"  ⚠ SeparateMember 혼입: [{row['variable']}] {row['period']} — {ctx}")
if sep_count == 0:
    print("  ✓ SeparateMember 행 0건 — 필터 정상 작동")

# v2 보정 효과 요약
print(f"\n── v2 보정 효과 요약 ──")
print("  [1] Dimension축: PlanAssetsMember -15 / ObligationMember +5 (변수별 방향)")
print("  [2] DiscountRate: ActuarialAssumption +5 / CashFlowProjection -15")
print("  [3] DBO 변수: '순' 레이블 -10 (기존 유지)")
print("  [4] D섹션: Dimension 없는 행 우선 (총계 우선)")

print(f"\n{'='*80}")
print("진단 완료. 검토 항목은 사용자 수동 판단 필요.")
print(f"{'='*80}")

CIK 00101044 강화 진단 — ConsolidatedMember 엄격 필터 + 3개년 × 13변수 (v2)
  2024.12 (2024_4Q 당기): ✓ 데이터 존재
  2023.12 (2024_4Q 전기 비교재무): ✓ 데이터 존재
  2022.12 (2023_4Q 전기 비교재무): ✗ CIK 미존재

────────────────────────────────────────────────────────────────────────────────
[B] 변수별 매핑 결과 (v2: Dimension축 + EID 보정 적용)
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
[C] 요약 매트릭스 — 13변수 × 3개년 (v2)
────────────────────────────────────────────────────────────────────────────────

CIK: 00101044
기간 소스: 2024.12(2024_4Q CFY) | 2023.12(2024_4Q PFY) | 2022.12(2023_4Q PFY)



,구분,BS/PL,ELEMENT_ID,한글레이블,2024.12,2024.12_상태,2023.12,2023.12_상태,2022.12,2022.12_상태
DBO,★필수,eFY,ifrs-full_DefinedBenefitObligationAtPresentValue,"확정급여채무, 현재가치",140.4억,주의,116.9억,주의,-,데이터없음
PlanAsset,★필수,eFY,ifrs-full_PlanAssetsAtFairValue,"사외적립자산, 공정가치",38.0억,정상,39.6억,정상,-,데이터없음
NetDBO,○선택,eFY,ifrs-full_LiabilityAssetOfDefinedBenefitPlans,순확정급여부채(자산),140.4억,주의,77.3억,주의,-,데이터없음
DiscountRate,★필수,eFY,ifrs-full_ActuarialAssumptionOfDiscountRates,할인율에 대한 보험수리적 가정,3.29%,주의,3.86%,주의,-,데이터없음
SalaryGrowth,★필수,eFY,ifrs-full_ActuarialAssumptionOfExpectedRatesOf...,미래임금상승률에 대한 보험수리적 가정,3.61%,주의,4.00%,주의,-,데이터없음
Duration,★필수,eFY,-,-,-,매핑없음,-,매핑없음,-,데이터없음
ServiceCost,★필수,dFY,ifrs-full_CurrentServiceCostNetDefinedBenefitL...,"당기근무원가, 순확정급여부채(자산)",22.0억,주의,22.2억,주의,-,데이터없음
InterestCost,★필수,dFY,ifrs-full_InterestExpenseIncomeNetDefinedBenef...,"이자비용(수익), 순확정급여부채(자산)",3.4억,주의,3.8억,주의,-,데이터없음
InterestIncome,○선택,dFY,ifrs-full_InterestIncomeDefinedBenefitPlans,"이자수익, 확정급여제도",1.6억,정상,1.8억,정상,-,데이터없음
BenefitPayment,○선택,dFY,ifrs-full_PaymentsFromPlanNetDefinedBenefitLia...,"제도에서 지급한 금액, 순확정급여부채(자산)",12.7억,주의,14.9억,주의,-,데이터없음



2024.12 상태: {'주의': np.int64(9), '정상': np.int64(2), '매핑없음': np.int64(2)}

2023.12 상태: {'주의': np.int64(8), '정상': np.int64(2), '매핑없음': np.int64(2), '검토': np.int64(1)}

2022.12 상태: {'데이터없음': np.int64(13)}

────────────────────────────────────────────────────────────────────────────────
[D] DBO 총액/순액 판별
────────────────────────────────────────────────────────────────────────────────

  IFRS DBO (총액): 14,042,701,488 (ifrs-full_DefinedBenefitObligationAtPresentValue)
  사외적립자산 (PA): 3,796,462,445 (ifrs-full_PlanAssetsAtFairValue)
  매핑 DBO 값: 14,042,701,488 (ifrs-full_DefinedBenefitObligationAtPresentValue)
  매핑 NetDBO 값: 14,042,701,488

  ▶ DBO 유형: 총액(DBO매핑≈IFRS_DBO)
    - IFRS DBO(14,042,701,488) + PA(3,796,462,445) 별도 존재
    - 계산 Net = DBO - PA = 10,246,239,043
    - 매핑DBO ≈ IFRS_DBO (차이 0.00%)

────────────────────────────────────────────────────────────────────────────────
[E] 이상 항목 상세 — 수동 검토 대상 (v2)
────────────────────────────────────────────────────────────────────────────────

전체 상태 

In [15]:
# =============================================================================
# Cell 4.6: 단일 CIK 전체 val + label 머징 테이블 (DART 보고서 대조용)
# =============================================================================
# val.tsv 원본에서 직접 로드 (퇴직연금 키워드 필터 없음)

TARGET_CIK = '00102751'  # 포스코이앤씨
q = '2024_4Q'

# val.tsv 원본에서 해당 CIK 직접 필터
vpath = BASE / q / 'val.tsv'
df_val_raw = pd.read_csv(vpath, sep='\t', dtype=str)
cik_val = df_val_raw[df_val_raw['CIK'] == TARGET_CIK].copy()
print(f"CIK {TARGET_CIK} 전체 val 행: {len(cik_val)}행")
print(f"  (참고: val_filtered는 {len(val_filtered[q][val_filtered[q]['CIK'] == TARGET_CIK])}행 — 퇴직연금 키워드 필터 적용)")

# lab_std 조인 (ELEMENT_ID = ELMT_ID)
cik_lab = lab_std[q][lab_std[q]['CIK'] == TARGET_CIK][['ELMT_ID', 'LABEL']].drop_duplicates()
cik_merged = cik_val.merge(cik_lab, left_on='ELEMENT_ID', right_on='ELMT_ID', how='left')

# VALUE 숫자 변환 + 억원 환산
cik_merged['VALUE_num'] = pd.to_numeric(cik_merged['VALUE'], errors='coerce')
cik_merged['VALUE_억'] = cik_merged['VALUE_num'] / 1e8

# 연결/별도 구분 컬럼
def ctx_type(ctx):
    if pd.isna(ctx): return ''
    if 'SeparateMember' in ctx: return '별도'
    if 'ConsolidatedMember' in ctx: return '연결'
    return '-'

cik_merged['연결구분'] = cik_merged['CONTEXT_ID'].apply(ctx_type)

# 기간 구분
def ctx_period(ctx):
    if pd.isna(ctx): return ''
    if 'CFY' in ctx: return 'CFY(당기)'
    if 'PFY' in ctx: return 'PFY(전기)'
    if 'BPFY' in ctx: return 'BPFY(전전기)'
    return ctx[:15]

cik_merged['기간'] = cik_merged['CONTEXT_ID'].apply(ctx_period)

# BS/PL 구분
def ctx_bs_pl(ctx):
    if pd.isna(ctx): return ''
    if 'eFY' in ctx: return 'BS(기말)'
    if 'dFY' in ctx: return 'PL(기간)'
    return '-'

cik_merged['BS_PL'] = cik_merged['CONTEXT_ID'].apply(ctx_bs_pl)

# 퇴직연금 관련 여부 표시
pension_mask = cik_merged['ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False)
cik_merged['퇴직연금'] = pension_mask.map({True: '●', False: ''})

# 표시 컬럼 정리 + 정렬
cols = ['퇴직연금', 'LABEL', 'ELEMENT_ID', '연결구분', '기간', 'BS_PL', 'UNIT_ID', 'VALUE_억', 'VALUE', 'CONTEXT_ID']
display(
    cik_merged[cols]
    .sort_values(['연결구분', '기간', 'ELEMENT_ID'])
    .reset_index(drop=True)
    .style.format({'VALUE_억': '{:,.1f}'})
)

print(f"\n퇴직연금 키워드 매칭: {pension_mask.sum()}행 / 전체 {len(cik_merged)}행")

cik_merged[cols].to_excel(f'CIK_{TARGET_CIK}_val_label_merged.xlsx', index=False)

CIK 00102751 전체 val 행: 1269행
  (참고: val_filtered는 30행 — 퇴직연금 키워드 필터 적용)


,퇴직연금,LABEL,ELEMENT_ID,연결구분,기간,BS_PL,UNIT_ID,VALUE_억,VALUE,CONTEXT_ID
0,,nan,dart-gcd_Amendment,-,CFY(당기),PL(기간),nan,nan,false,CFY2024dFY_dart-gcd_PeriodAxis_dart-gcd_PeriodCoveredbytheReportofCurrentFiscalYearorQuarterorHalfYearMember
1,,nan,dart-gcd_AuthorName,-,CFY(당기),PL(기간),nan,nan,Park ki-cheon,CFY2024dFY_dart-gcd_AuthorInformationAxis_dart-gcd_PersonnelInChargeMember
2,,nan,dart-gcd_AuthorName,-,CFY(당기),PL(기간),nan,nan,Jeong duk-kwan,CFY2024dFY_dart-gcd_AuthorInformationAxis_dart-gcd_PersonnelForContactMember
3,,nan,dart-gcd_AuthorName,-,CFY(당기),PL(기간),nan,nan,자체작성,CFY2024dFY_dart-gcd_AuthorInformationAxis_dart-gcd_XBRLConsultingFirmMember
4,,nan,dart-gcd_AuthorTitle,-,CFY(당기),PL(기간),nan,nan,Executive Vice President,CFY2024dFY_dart-gcd_AuthorInformationAxis_dart-gcd_PersonnelInChargeMember
5,,nan,dart-gcd_AuthorTitle,-,CFY(당기),PL(기간),nan,nan,Section Chief,CFY2024dFY_dart-gcd_AuthorInformationAxis_dart-gcd_PersonnelForContactMember
6,,nan,dart-gcd_ConsolidatedAuditOpinion,-,CFY(당기),PL(기간),nan,nan,Unqualified Opinion,CFY2024dFY_dart-gcd_PeriodAxis_dart-gcd_PeriodCoveredbytheReportofCurrentFiscalYearorQuarterorHalfYearMember
7,,nan,dart-gcd_ConsolidatedAuditOpinion,-,CFY(당기),PL(기간),nan,nan,Unqualified Opinion,CFY2024dFY_dart-gcd_PeriodAxis_dart-gcd_PeriodCoveredbyLastFiscalYearMember
8,,nan,dart-gcd_ConsolidatedAuditOpinion,-,CFY(당기),PL(기간),nan,nan,Unqualified Opinion,CFY2024dFY_dart-gcd_PeriodAxis_dart-gcd_PeriodCoveredbyTheYearBeforeLastFiscalYearMember
9,,nan,dart-gcd_ConsolidatedAuditReportDate,-,CFY(당기),PL(기간),nan,nan,2025-03-12,CFY2024dFY_dart-gcd_PeriodAxis_dart-gcd_PeriodCoveredbytheReportofCurrentFiscalYearorQuarterorHalfYearMember



퇴직연금 키워드 매칭: 30행 / 전체 1269행


In [16]:
# =============================================================================
# Cell 5: CIK별 ELEMENT_ID 매핑 테이블 (Step 3) — v2
# =============================================================================
# 변경사항 v2:
#   1. UNIT_ID 필터 추가 (expected_unit과 불일치 시 제외)
#   2. DBO 변수: ko_label에 '순' 포함 시 penalty (-10)
#   3. ELEMENT_ID 우선순위 점수 추가 (ifrs > dart > entity)
#   4. 이상치 플래그 컬럼 추가

# --- VALUE 이상치 보정 (DECIMALS 오류 기업 하드코딩) ---
# 아세아제지(00138729): 2024_4Q DECIMALS=-6이나 VALUE가 10^6 과대 저장 (DART 파일링 오류)
# 보정: VALUE × 10^DECIMALS 적용
VALUE_OUTLIER_CIKS = {
    '2024_4Q': {'00138729'},   # 아세아제지
    # '2023_4Q': {},  ← 2023에는 해당 없음
}

for q_name, cik_set in VALUE_OUTLIER_CIKS.items():
    if q_name in val_filtered and cik_set:
        vf_q = val_filtered[q_name]
        mask = vf_q['CIK'].isin(cik_set) & vf_q['DECIMALS'].notna()
        if mask.any():
            dec = pd.to_numeric(vf_q.loc[mask, 'DECIMALS'], errors='coerce')
            dec = dec.where(dec.abs() < 100, 0).fillna(0).astype(int)  # INF, NaN → 0
            orig_val = pd.to_numeric(vf_q.loc[mask, 'VALUE'], errors='coerce')
            corrected = orig_val * (10.0 ** dec)
            vf_q.loc[mask, 'VALUE'] = corrected.astype(str)
            n_fixed = mask.sum()
            print(f"VALUE 이상치 보정: {q_name} {cik_set} → {n_fixed}행 보정 완료")


def compute_ctx_score(ctx_series):
    """CONTEXT_ID에서 연결/당기/기말 점수를 벡터화 계산"""
    ctx = ctx_series.fillna('')
    is_con = (~ctx.str.contains('SeparateMember', na=False)) | ctx.str.contains('ConsolidatedMember', na=False)
    is_cur = ctx.str.contains('CFY', na=False)
    is_end = ctx.str.contains('eFY', na=False)
    return is_con.astype(int) * 4 + is_cur.astype(int) * 2 + is_end.astype(int)

def compute_eid_priority(eid_series):
    """ELEMENT_ID prefix 우선순위: ifrs-full(+8) > dart(+4) > entity(+0)"""
    eid = eid_series.fillna('')
    score = pd.Series(0, index=eid.index)
    score[eid.str.startswith('ifrs-full_')] = 8
    score[eid.str.startswith('dart_')] = 4
    return score

# lab_std → (CIK, ELMT_ID) → LABEL 딕셔너리
lab_lookup = {}
for q in QUARTERS:
    lab_lookup[q] = lab_std[q].drop_duplicates(subset=['CIK', 'ELMT_ID']).set_index(['CIK', 'ELMT_ID'])['LABEL'].to_dict()

mapping_frames = []

for q in QUARTERS:
    vf = val_filtered[q]
    print(f"[{q}] 퇴직연금 관련 데이터 보유 CIK: {vf['CIK'].nunique():,}개")
    
    for var_name, var_def in VARIABLES.items():
        if var_name not in var_candidates or len(var_candidates[var_name]) == 0:
            continue
        cand_eids = set(var_candidates[var_name][var_candidates[var_name]['quarter'] == q]['ELEMENT_ID'])
        if not cand_eids:
            continue
        
        # val에서 해당 ELEMENT_ID 행 필터
        subset = vf[vf['ELEMENT_ID'].isin(cand_eids)].copy()
        if len(subset) == 0:
            continue
        
        # UNIT_ID 필터: 기대 UNIT과 일치하는 행만 (NaN은 유지)
        expected_unit = var_def['expected_unit']
        unit_mask = (subset['UNIT_ID'] == expected_unit) | subset['UNIT_ID'].isna()
        subset = subset[unit_mask].copy()
        if len(subset) == 0:
            continue
        
        # 복합 스코어: ctx(0~7) + eid_priority(0~8) + label_penalty
        subset['_ctx_score'] = compute_ctx_score(subset['CONTEXT_ID'])
        subset['_eid_score'] = compute_eid_priority(subset['ELEMENT_ID'])
        subset['_score'] = subset['_ctx_score'] + subset['_eid_score']
        
        # DBO 변수: 한글 레이블에 '순' 포함 시 penalty (-10)
        if var_name == 'DBO':
            lkup = lab_lookup[q]
            labels = subset.apply(lambda r: lkup.get((r['CIK'], r['ELEMENT_ID']), ''), axis=1)
            net_penalty = labels.str.contains('순확정|순 확정', na=False).astype(int) * (-10)
            subset['_score'] = subset['_score'] + net_penalty
        
        # CIK별 최고 스코어 행 선택
        best_idx = subset.groupby('CIK')['_score'].idxmax()
        best = subset.loc[best_idx].copy()
        
        # 한글 레이블 조회
        lkup = lab_lookup[q]
        best['ko_label'] = best.apply(lambda r: lkup.get((r['CIK'], r['ELEMENT_ID']), ''), axis=1)
        best['variable'] = var_name
        best['quarter'] = q
        
        # 이상치 플래그
        flags = []
        for _, row in best.iterrows():
            f = []
            if row['UNIT_ID'] != expected_unit and pd.notna(row['UNIT_ID']):
                f.append(f'UNIT:{row["UNIT_ID"]}')
            if var_name == 'DBO' and '순' in str(row['ko_label']):
                f.append('NET_LABEL')
            if str(row['ELEMENT_ID']).startswith('entity'):
                f.append('ENTITY')
            flags.append('|'.join(f) if f else '')
        best['flag'] = flags
        
        mapping_frames.append(best[['CIK', 'REPORT_DATE', 'variable', 'ELEMENT_ID',
                                     'TAXONOMY_ID', 'ko_label', 'UNIT_ID', 'CONTEXT_ID',
                                     'VALUE', 'quarter', 'flag']].rename(columns={'VALUE': 'sample_VALUE'}))

df_mapping = pd.concat(mapping_frames, ignore_index=True)
print(f"\n전체 매핑: {len(df_mapping):,} rows")
print(f"고유 CIK: {df_mapping['CIK'].nunique():,}개")
print(f"\n변수별 매핑 수:")
print(df_mapping.groupby('variable')['CIK'].nunique().sort_values(ascending=False).to_string())

# --- 이상치 요약 ---
flagged = df_mapping[df_mapping['flag'] != '']
if len(flagged) > 0:
    print(f"\n이상치 플래그 요약 ({len(flagged):,}건):")
    print(flagged['flag'].value_counts().head(20).to_string())

# --- 연도 간 ELEMENT_ID 변동 확인 ---
print(f"\n{'='*60}")
print("연도 간 동일 CIK의 ELEMENT_ID 변동 분석")
print(f"{'='*60}")

m24 = df_mapping[df_mapping['quarter'] == '2024_4Q'][['CIK', 'variable', 'ELEMENT_ID']]
m23 = df_mapping[df_mapping['quarter'] == '2023_4Q'][['CIK', 'variable', 'ELEMENT_ID']]
merged = m24.merge(m23, on=['CIK', 'variable'], suffixes=('_24', '_23'))

if len(merged) > 0:
    changed = merged[merged['ELEMENT_ID_24'] != merged['ELEMENT_ID_23']]
    print(f"비교 가능: {len(merged):,}건, 변동: {len(changed):,}건 ({len(changed)/len(merged)*100:.1f}%)")
    if len(changed) > 0:
        print(f"\n변동 케이스 (상위 20):")
        for _, row in changed.head(20).iterrows():
            print(f"  CIK={row['CIK']} [{row['variable']}]: {row['ELEMENT_ID_23']} → {row['ELEMENT_ID_24']}")
else:
    print("비교 가능한 데이터 없음")

# CSV 저장
csv_path = BASE / 'pension_cik_mapping.csv'
df_mapping.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"\n저장: {csv_path}")


VALUE 이상치 보정: 2024_4Q {'00138729'} → 219행 보정 완료
[2024_4Q] 퇴직연금 관련 데이터 보유 CIK: 2,350개
[2023_4Q] 퇴직연금 관련 데이터 보유 CIK: 2,245개

전체 매핑: 18,109 rows
고유 CIK: 2,398개

변수별 매핑 수:
variable
NetDBO                   2255
DBO                      2134
ActuarialGL              2131
PlanAsset                 887
BenefitPayment            737
ServiceCost               473
InterestCost              472
DiscountRate              468
SalaryGrowth              466
RetirementBenefitCost     449
Duration                  379
InterestIncome            147
NetInterest                 4

이상치 플래그 요약 (1,580건):
flag
NET_LABEL           1056
ENTITY               400
NET_LABEL|ENTITY     124

연도 간 동일 CIK의 ELEMENT_ID 변동 분석
비교 가능: 7,107건, 변동: 1,739건 (24.5%)

변동 케이스 (상위 20):
  CIK=00102140 [DBO]: dart_PresentValueOfDefinedBenefitObligation → dart_PostemploymentBenefitObligations
  CIK=00102432 [DBO]: dart_PostemploymentBenefitObligations → ifrs-full_DefinedBenefitObligationAtPresentValue
  CIK=00103006 [DBO]: dart_Poste

In [17]:
# =============================================================================
# Cell 6: DBO 유형 판별 (Step 4 — 핵심) — v2 전체 CIK 대상
# =============================================================================
# 변경사항 v2:
#   1. dart_PostemploymentBenefitObligations 사용자 뿐 아니라 전체 DBO/NetDBO CIK 대상
#   2. 레이블 정규화 (공백/번호 제거)
#   3. '확정급여부채' 레이블 → 추가 검증 로직 (판별불가 축소)
#   4. 음수값, 0값, 비정상 레이블 플래그 추가

q = '2024_4Q'
vf = val_filtered[q]
ls = lab_std[q]

# --- cal.tsv 로드 ---
df_cal = pd.read_csv(BASE / q / 'cal.tsv', sep='\t', dtype=str,
                     usecols=['CIK', 'REPORT_DATE', 'ELEMENT_ID', 'TAXONOMY_ID',
                              'PARENT_ELEMENT_ID', 'WEIGHT'])
print(f"cal.tsv: {len(df_cal):,} rows")

cal_pension = df_cal[
    df_cal['ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False) |
    df_cal['PARENT_ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False)
].copy()
print(f"퇴직연금 관련 cal 행: {len(cal_pension):,}")

if len(cal_pension) > 0:
    print("\n주요 PARENT→CHILD 계산관계 (상위 20):")
    cal_summary = cal_pension.groupby(['PARENT_ELEMENT_ID', 'ELEMENT_ID', 'WEIGHT']).size().reset_index(name='n_cik')
    cal_summary = cal_summary.sort_values('n_cik', ascending=False)
    for _, row in cal_summary.head(20).iterrows():
        w = row['WEIGHT']
        sign = '+' if w in ('1', '1.0') else '-' if w in ('-1', '-1.0') else f'({w})'
        print(f"  {sign} {row['ELEMENT_ID'][:60]:60s} ← {row['PARENT_ELEMENT_ID'][:40]:40s} [{row['n_cik']}개사]")

del df_cal
gc.collect()

# --- cal.tsv에서 DBO-PA=Net 관계 확인 CIK 세트 ---
cal_net_ciks = set()
if len(cal_pension) > 0:
    net_parents = cal_pension[
        cal_pension['PARENT_ELEMENT_ID'].str.contains('NetDefinedBenefit|PostemploymentBenefit', case=False, na=False)
    ]
    for cik in net_parents['CIK'].unique():
        children = net_parents[net_parents['CIK'] == cik]
        has_dbo_child = children['ELEMENT_ID'].str.contains('DefinedBenefitObligation', case=False, na=False).any()
        has_pa_child = children['ELEMENT_ID'].str.contains('PlanAsset|FairValue', case=False, na=False).any()
        if has_dbo_child or has_pa_child:
            cal_net_ciks.add(cik)
print(f"\ncal.tsv DBO-PA=Net 관계 확인 CIK: {len(cal_net_ciks)}개")

# --- 대상 CIK: DBO 또는 NetDBO 변수에 매핑된 전체 CIK ---
m24 = df_mapping[df_mapping['quarter'] == q]
dbo_ciks = set(m24[m24['variable'] == 'DBO']['CIK'])
net_ciks = set(m24[m24['variable'] == 'NetDBO']['CIK'])
all_target_ciks = sorted(dbo_ciks | net_ciks)
print(f"\nDBO 유형 판별 대상 CIK: {len(all_target_ciks)}개 (DBO: {len(dbo_ciks)}, NetDBO: {len(net_ciks)})")

# IFRS 표준 ELEMENT_ID 목록
ifrs_dbo_eids = [
    'ifrs-full_DefinedBenefitObligationAtPresentValue',
    'ifrs-full_PresentValueOfDefinedBenefitObligation',
]
ifrs_pa_eids = [
    'ifrs-full_PlanAssetsAtFairValue',
    'ifrs-full_FairValueOfPlanAssets',
]

# 레이블 정규화 함수
def normalize_label(text):
    """번호, 공백, 특수문자 제거"""
    if not isinstance(text, str):
        return ''
    text = re.sub(r'^[\d\.\(\)\s]+', '', text)  # 앞의 번호/괄호 제거
    text = text.strip().replace(' ', '')
    return text

# --- DBO 유형 판별 (전체 CIK) ---
dbo_type_results = []

for cik in all_target_ciks:
    cik_data = vf[vf['CIK'] == cik]
    cik_labels = ls[ls['CIK'] == cik]
    
    # DBO 매핑에서 사용된 ELEMENT_ID와 값
    dbo_row = m24[(m24['CIK'] == cik) & (m24['variable'] == 'DBO')]
    net_row = m24[(m24['CIK'] == cik) & (m24['variable'] == 'NetDBO')]
    
    dbo_eid = dbo_row.iloc[0]['ELEMENT_ID'] if len(dbo_row) > 0 else ''
    dbo_label_raw = dbo_row.iloc[0]['ko_label'] if len(dbo_row) > 0 else ''
    dbo_label = normalize_label(dbo_label_raw)
    
    dbo_val = None
    dbo_decimals = None
    if len(dbo_row) > 0:
        try:
            dbo_val = float(dbo_row.iloc[0]['sample_VALUE'])
            dbo_decimals = dbo_row.iloc[0].get('DECIMALS', '')
        except (ValueError, TypeError):
            pass
    
    net_eid = net_row.iloc[0]['ELEMENT_ID'] if len(net_row) > 0 else ''
    net_val = None
    if len(net_row) > 0:
        try:
            net_val = float(net_row.iloc[0]['sample_VALUE'])
        except (ValueError, TypeError):
            pass
    
    # IFRS DBO 총액 존재 여부
    has_ifrs_dbo = cik_data['ELEMENT_ID'].isin(ifrs_dbo_eids).any()
    ifrs_dbo_val = None
    if has_ifrs_dbo:
        ifrs_rows = cik_data[cik_data['ELEMENT_ID'].isin(ifrs_dbo_eids)]
        # 연결 당기 우선
        ctx_scores = compute_ctx_score(ifrs_rows['CONTEXT_ID'])
        best_row = ifrs_rows.iloc[ctx_scores.argmax()]
        try:
            ifrs_dbo_val = float(best_row['VALUE'])
        except (ValueError, TypeError):
            pass
    
    # 사외적립자산 존재 여부
    has_pa = cik_data['ELEMENT_ID'].isin(ifrs_pa_eids).any()
    pa_val = None
    if has_pa:
        pa_rows = cik_data[cik_data['ELEMENT_ID'].isin(ifrs_pa_eids)]
        ctx_scores = compute_ctx_score(pa_rows['CONTEXT_ID'])
        best_row = pa_rows.iloc[ctx_scores.argmax()]
        try:
            pa_val = float(best_row['VALUE'])
        except (ValueError, TypeError):
            pass
    
    has_dim = cik_data['CONTEXT_ID'].str.contains('NetDefinedBenefitLiabilityAssetAxis', na=False).any()
    has_cal_net = cik in cal_net_ciks
    
    # ---- 유형 판별 로직 (우선순위) ----
    anomaly = []
    
    # Case 1: IFRS DBO + PA 모두 존재 → 총액 분리 가능
    if has_ifrs_dbo and has_pa:
        dbo_type = '총액분리'
        # 교차 검증: DBO 매핑값 ≈ IFRS DBO인지?
        if dbo_val is not None and ifrs_dbo_val is not None and pa_val is not None:
            net_calc = ifrs_dbo_val - pa_val
            if net_calc != 0 and dbo_val != 0:
                ratio = dbo_val / net_calc
                if 0.95 <= abs(ratio) <= 1.05:
                    dbo_type = '순액(DBO매핑=Net)'
                    anomaly.append('DBO_IS_NET')
                elif ifrs_dbo_val != 0 and abs(dbo_val - ifrs_dbo_val) / abs(ifrs_dbo_val) < 0.05:
                    dbo_type = '총액(DBO매핑≈IFRS_DBO)'
    
    # Case 2: IFRS DBO만 존재 (PA 없음)
    elif has_ifrs_dbo and not has_pa:
        dbo_type = '총액(PA없음)'
    
    # Case 3: Dimension 분리 (NetDefinedBenefitLiabilityAssetAxis)
    elif has_dim:
        dbo_type = 'Dimension분리형'
    
    # Case 4: cal.tsv에서 Net=DBO-PA 관계 확인
    elif has_cal_net:
        dbo_type = '순액(cal검증)'
    
    # Case 5: 레이블 기반 판별
    elif '순확정급여' in dbo_label:
        dbo_type = '순액(레이블)'
    elif '확정급여채무' in dbo_label:
        dbo_type = '총액추정(레이블)'
    elif '퇴직급여부채' in dbo_label:
        dbo_type = '순액추정(퇴직급여부채)'
    elif '종업원급여' in dbo_label or '충당' in dbo_label:
        dbo_type = '오분류의심'
        anomaly.append('WRONG_LABEL')
    elif dbo_eid == '' and net_eid != '':
        dbo_type = 'NetDBO만'
    else:
        dbo_type = '판별불가'
    
    # 추가 이상치 검출
    if dbo_val is not None:
        if dbo_val == 0:
            anomaly.append('ZERO_VAL')
        if dbo_val < 0:
            anomaly.append('NEGATIVE_VAL')
    
    dbo_type_results.append({
        'CIK': cik,
        'dbo_eid': dbo_eid,
        'dbo_label': dbo_label,
        'dbo_label_raw': dbo_label_raw,
        'dbo_val': dbo_val,
        'net_eid': net_eid,
        'net_val': net_val,
        'has_ifrs_dbo': has_ifrs_dbo,
        'ifrs_dbo_val': ifrs_dbo_val,
        'has_pa': has_pa,
        'pa_val': pa_val,
        'has_dim': has_dim,
        'has_cal_net': has_cal_net,
        'dbo_type': dbo_type,
        'anomaly': '|'.join(anomaly) if anomaly else '',
    })

df_dbo_type = pd.DataFrame(dbo_type_results)
print(f"\n{'='*60}")
print(f"DBO 유형 판별 결과 (2024_4Q) — 전체 CIK 대상")
print(f"{'='*60}")
print(f"전체: {len(df_dbo_type)}개사")
print(f"\n유형별 분포:")
print(df_dbo_type['dbo_type'].value_counts().to_string())

print(f"\n이상치 분포:")
anomaly_counts = df_dbo_type[df_dbo_type['anomaly'] != '']['anomaly'].value_counts()
if len(anomaly_counts) > 0:
    print(anomaly_counts.to_string())
else:
    print("  없음")

print(f"\n정규화 레이블 분포 (상위 15):")
print(df_dbo_type['dbo_label'].value_counts().head(15).to_string())

# 유형별 샘플
for dtype in df_dbo_type['dbo_type'].unique():
    subset = df_dbo_type[df_dbo_type['dbo_type'] == dtype]
    print(f"\n--- {dtype} ({len(subset)}개사) 샘플 ---")
    for _, row in subset.head(3).iterrows():
        print(f"  CIK={row['CIK']}: dbo={row['dbo_val']}, ifrs={row['ifrs_dbo_val']}, "
              f"pa={row['pa_val']}, label='{row['dbo_label']}', anomaly={row['anomaly']}")

del cal_pension


cal.tsv: 578,632 rows
퇴직연금 관련 cal 행: 11,591

주요 PARENT→CHILD 계산관계 (상위 20):
  + ifrs-full_OtherComprehensiveIncomeNetOfTaxGainsLossesOnRemea ← ifrs-full_OtherComprehensiveIncomeThatWi [2859개사]
  + dart_PostemploymentBenefitObligations                        ← ifrs-full_NoncurrentLiabilities          [1797개사]
  + dart_AdjustmentsForProvisionForSeveranceIndemnities          ← ifrs-full_AdjustmentsForReconcileProfitL [870개사]
  + dart_InvestedAssetForPostemploymentBenefit                   ← ifrs-full_NoncurrentAssets               [597개사]
  + ifrs-full_NoncurrentRecognisedLiabilitiesDefinedBenefitPlan  ← ifrs-full_NoncurrentLiabilities          [475개사]
  + ifrs-full_NoncurrentRecognisedAssetsDefinedBenefitPlan       ← ifrs-full_NoncurrentAssets               [457개사]
  + dart_AdjustmentsForDecreaseincreaseInFairValueOfPlanAssets   ← dart_AdjustmentsForAssetsLiabilitiesOfOp [456개사]
  + dart_AdjustmentsForIncreasedecreaseInPostemploymentBenefitOb ← dart_AdjustmentsForAssetsLiabilitiesOfOp [44

In [18]:
# =============================================================================
# Cell 7: CONTEXT_ID 패턴 분석 + def.tsv Dimension 구조
# =============================================================================

q = '2024_4Q'
vf = val_filtered[q]

# CONTEXT_ID 전체 구조 파싱
ctx_ids = vf['CONTEXT_ID'].dropna().unique()
print(f"퇴직연금 관련 고유 CONTEXT_ID: {len(ctx_ids):,}개")

# --- 1. 시점 prefix 분석 ---
prefix_pattern = re.compile(r'^((?:BP)?[CP]?FY\d{4}[de]FY)')
prefixes = []
for ctx in ctx_ids:
    m = prefix_pattern.match(ctx)
    prefixes.append(m.group(1) if m else ctx.split('_')[0])

print(f"\n시점 prefix 분포 (상위 20):")
print(pd.Series(prefixes).value_counts().head(20).to_string())

# --- 2. 연결/별도 구분 ---
con_sep = []
for ctx in ctx_ids:
    if 'ConsolidatedMember' in ctx:
        con_sep.append('Consolidated')
    elif 'SeparateMember' in ctx:
        con_sep.append('Separate')
    else:
        con_sep.append('None/Default')
print(f"\n연결/별도 구분:")
print(pd.Series(con_sep).value_counts().to_string())

# --- 3. UNIT_ID 분포 (가이드: KRW, SHARES, KRWEPS, PURE, USD) ---
print(f"\nUNIT_ID 분포 (퇴직연금 필터 데이터):")
print(vf['UNIT_ID'].value_counts().to_string())

# PURE 단위 = 비율 변수 (할인율, 임금상승률 등)
pure_data = vf[vf['UNIT_ID'] == 'PURE']
if len(pure_data) > 0:
    print(f"\nPURE 단위 사용 ELEMENT_ID (비율 변수):")
    print(pure_data['ELEMENT_ID'].value_counts().head(15).to_string())

# --- 4. DECIMALS 분포 ---
print(f"\nDECIMALS 분포:")
print(vf['DECIMALS'].value_counts().head(15).to_string())

# --- 5. def.tsv 로드 — Dimension 구조 분석 ---
print(f"\n{'='*60}")
print("def.tsv Dimension 구조 분석")

df_def = pd.read_csv(BASE / q / 'def.tsv', sep='\t', dtype=str,
                     usecols=['CIK', 'REPORT_DATE', 'ROLE_ID', 'ELEMENT_ID',
                              'TAXONOMY_ID', 'PARENT_ELEMENT_ID', 'ARCROLE'])
print(f"def.tsv: {len(df_def):,} rows")

# 퇴직연금 관련 def 행 필터
def_pension = df_def[
    df_def['ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False) |
    df_def['PARENT_ELEMENT_ID'].str.contains(PENSION_KEYWORDS_EN, case=False, na=False)
]
print(f"퇴직연금 관련 def 행: {len(def_pension):,}")

# ARCROLE 분포 (hypercube-dimension, dimension-domain, domain-member 등)
if len(def_pension) > 0:
    print(f"\nARCROLE 분포:")
    print(def_pension['ARCROLE'].value_counts().to_string())

    # Dimension 축 구조: hypercube → dimension → domain → member
    # dimension 유형만 추출
    dim_rows = def_pension[def_pension['ARCROLE'].str.contains('dimension', case=False, na=False)]
    if len(dim_rows) > 0:
        print(f"\nDimension 축 (Hypercube→Dimension 관계):")
        dim_summary = dim_rows.groupby(['PARENT_ELEMENT_ID', 'ELEMENT_ID']).size().reset_index(name='n_cik')
        dim_summary = dim_summary.sort_values('n_cik', ascending=False)
        for _, row in dim_summary.head(15).iterrows():
            print(f"  {row['PARENT_ELEMENT_ID'][:50]:50s} → {row['ELEMENT_ID'][:50]:50s} [{row['n_cik']}개사]")

    # domain-member 관계 (Member 값 후보)
    member_rows = def_pension[def_pension['ARCROLE'].str.contains('domain-member', case=False, na=False)]
    if len(member_rows) > 0:
        print(f"\nDomain→Member 관계 (상위 20):")
        mem_summary = member_rows.groupby(['PARENT_ELEMENT_ID', 'ELEMENT_ID']).size().reset_index(name='n_cik')
        mem_summary = mem_summary.sort_values('n_cik', ascending=False)
        for _, row in mem_summary.head(20).iterrows():
            print(f"  {row['PARENT_ELEMENT_ID'][:50]:50s} → {row['ELEMENT_ID'][:50]:50s} [{row['n_cik']}개사]")

del df_def
gc.collect()

# --- 6. cntxt.tsv 조인으로 상세 확인 ---
print(f"\n{'='*60}")
print("cntxt.tsv 조인 분석")

df_ctx = pd.read_csv(BASE / q / 'cntxt.tsv', sep='\t', dtype=str)
print(f"cntxt.tsv: {len(df_ctx):,} rows")

pension_ctx_ids = set(vf['CONTEXT_ID'].dropna().unique())
ctx_pension = df_ctx[df_ctx['CONTEXT_ID'].isin(pension_ctx_ids)]
print(f"퇴직연금 관련 cntxt 행: {len(ctx_pension):,}")

if len(ctx_pension) > 0:
    # AXIS별 MEMBER 분포
    print(f"\nAXIS_ELEMENT_ID × MEMBER_ELEMENT_ID 주요 조합:")
    axis_member = ctx_pension.groupby(['AXIS_ELEMENT_ID', 'MEMBER_ELEMENT_ID']).size().reset_index(name='count')
    axis_member = axis_member.sort_values('count', ascending=False)
    print(axis_member.head(30).to_string(index=False))

    # DIMENSION 유형 (segment vs scenario)
    print(f"\nDIMENSION 유형 분포:")
    print(ctx_pension['DIMENSION'].value_counts().to_string())

    # 기간 정보 확인
    instant = ctx_pension[ctx_pension['PERIOD_INSTANT'].notna()]
    duration = ctx_pension[ctx_pension['PERIOD_START_DATE'].notna()]
    print(f"\ninstant 기간: {len(instant):,}건, duration 기간: {len(duration):,}건")

# --- 7. 필터 규칙 제안 ---
print(f"\n{'='*60}")
print("CONTEXT_ID 필터링 규칙 제안")
print(f"{'='*60}")
print("""
1. 연결재무제표 우선:
   - 'ConsolidatedMember' → 연결  |  'SeparateMember' → 별도
   - 둘 다 없으면 기본(별도 간주 — 연결 의무 없는 기업)

2. 당기 기말 시점:
   - BS 항목 (DBO, PA, NetDBO): CFY + eFY (instant)
   - PL 항목 (ServiceCost, Interest): CFY + dFY (duration)
   - PFY → 전기, BPFY → 전전기 (비교 재무)

3. Dimension 필터 (def.tsv 기반):
   - ConsolidatedAndSeparateFinancialStatementsAxis → 연결/별도
   - NetDefinedBenefitLiabilityAssetAxis → DBO/PA 총액 분리 (Member로 구분)
   - ActuarialAssumptionAxis → 보험수리적 가정 종류 구분

4. UNIT_ID 기반 변수 유형 구분:
   - KRW → 금액 (DBO, PA, ServiceCost, BenefitPayment 등)
   - PURE → 비율 (DiscountRate, SalaryGrowth, Duration 등)

5. DECIMALS 해석:
   - 0 → 원 단위  |  -3 → 천원 단위  |  -6 → 백만 단위
   - VALUE에 DECIMALS 적용 불필요 (이미 실제값 저장)
""")

del df_ctx
gc.collect()

퇴직연금 관련 고유 CONTEXT_ID: 4,901개

시점 prefix 분포 (상위 20):
CFY2024dFY     1380
PFY2023dFY     1089
CFY2024eFY     1066
PFY2023eFY      995
BPFY2022dFY     214
BPFY2022eFY      93
CFY2023dFY       18
BPFY2021dFY      17
PFY2022dFY       17
BPFY1eFY          6
PFY2022eFY        2
CFY2023eFY        2
BPFY2021eFY       2

연결/별도 구분:
Consolidated    2999
Separate        1902

UNIT_ID 분포 (퇴직연금 필터 데이터):
UNIT_ID
KRW     159721
PURE     13042
USD        241
JPY         28

PURE 단위 사용 ELEMENT_ID (비율 변수):
ELEMENT_ID
ifrs-full_ActuarialAssumptionOfDiscountRates                                                                                                                                                                                        2764
ifrs-full_ActuarialAssumptionOfExpectedRatesOfSalaryIncreases                                                                                                                                                                       2664
ifrs-full_PercentageOfReasonab

0

In [19]:
# =============================================================================
# Cell 8: 패턴 요약 리포트 (Step 5) — v2
# =============================================================================
# 변경사항 v2:
#   1. VARIABLES에서 expected_unit 직접 참조 (하드코딩 제거)
#   2. 민감도 변수 제외 반영
#   3. 필수 변수 목록 조정 (9→7, 민감도/RetirementBenefitCost/NetInterest 선택)
#   4. 이상치 플래그 통합 보고

print("="*80)
print("XBRL 퇴직연금 부채 인덱스 — ELEMENT_ID 패턴 요약 리포트 (v2)")
print("="*80)

# --- 1. 변수별 ELEMENT_ID 유형 분류표 ---
print("\n" + "─"*80)
print("1. 변수별 ELEMENT_ID 유형 분류")
print("─"*80)

report_rows = []
for var_name, var_def in VARIABLES.items():
    expected = var_def['expected_unit']
    
    if var_name not in var_candidates or len(var_candidates[var_name]) == 0:
        report_rows.append({
            '변수': var_name, '설명': var_def['desc'],
            'ifrs-full': 0, 'dart': 0, 'entity': 0,
            '총_ELEMENT_ID': 0, '최대_CIK수': 0,
            '기대_UNIT': expected,
            '대표_ELEMENT_ID': 'N/A',
        })
        continue
    
    cand = var_candidates[var_name]
    cand_24 = cand[cand['quarter'] == '2024_4Q']
    
    n_ifrs = cand_24[cand_24['type'] == 'ifrs-full']['ELEMENT_ID'].nunique()
    n_dart = cand_24[cand_24['type'] == 'dart']['ELEMENT_ID'].nunique()
    n_entity = cand_24[cand_24['type'] == 'entity']['ELEMENT_ID'].nunique()
    total_eid = cand_24['ELEMENT_ID'].nunique()
    max_cik = int(cand_24['n_cik'].max()) if len(cand_24) > 0 else 0
    
    # 대표 ELEMENT_ID: entity 제외, CIK수 최다
    best = cand_24[cand_24['type'] != 'entity'].sort_values('n_cik', ascending=False)
    best_eid = best.iloc[0]['ELEMENT_ID'] if len(best) > 0 else 'entity만'
    
    report_rows.append({
        '변수': var_name, '설명': var_def['desc'],
        'ifrs-full': n_ifrs, 'dart': n_dart, 'entity': n_entity,
        '총_ELEMENT_ID': total_eid, '최대_CIK수': max_cik,
        '기대_UNIT': expected,
        '대표_ELEMENT_ID': best_eid,
    })

df_report = pd.DataFrame(report_rows)
print(df_report.to_string(index=False))

# --- 2. CIK별 데이터 커버리지 매트릭스 ---
print("\n" + "─"*80)
print("2. CIK별 데이터 커버리지 매트릭스 (2024_4Q)")
print("─"*80)

REQUIRED_VARS = ['DBO', 'PlanAsset', 'DiscountRate', 'SalaryGrowth', 'Duration',
                 'ServiceCost', 'InterestCost']
OPTIONAL_VARS = ['BenefitPayment', 'ActuarialGL', 'RetirementBenefitCost', 'NetInterest', 'InterestIncome', 'NetDBO']

m24 = df_mapping[df_mapping['quarter'] == '2024_4Q']
cik_coverage = m24.pivot_table(index='CIK', columns='variable', values='ELEMENT_ID', aggfunc='first')

for v in REQUIRED_VARS + OPTIONAL_VARS:
    if v not in cik_coverage.columns:
        cik_coverage[v] = np.nan

coverage_count = cik_coverage[REQUIRED_VARS].notna().sum(axis=1)
print(f"\n필수 {len(REQUIRED_VARS)}개 변수 커버리지 분포:")
print(coverage_count.value_counts().sort_index().to_string())
print(f"\n전체 CIK: {len(cik_coverage)}개")
print(f"필수 변수 7/7 충족: {(coverage_count == len(REQUIRED_VARS)).sum()}개")
print(f"필수 변수 5개 이상: {(coverage_count >= 5).sum()}개")
print(f"필수 변수 3개 이상: {(coverage_count >= 3).sum()}개")

print(f"\n변수별 커버리지 (2024_4Q):")
for v in REQUIRED_VARS:
    n_has = cik_coverage[v].notna().sum()
    tag = '★필수'
    print(f"  {tag} {v:25s}: {n_has:>5,} / {len(cik_coverage):>5,} ({n_has/len(cik_coverage)*100:5.1f}%)")
for v in OPTIONAL_VARS:
    if v in cik_coverage.columns:
        n_has = cik_coverage[v].notna().sum()
        print(f"  ○선택 {v:25s}: {n_has:>5,} / {len(cik_coverage):>5,} ({n_has/len(cik_coverage)*100:5.1f}%)")

# --- 3. 이상치/주의 기업 통합 ---
print("\n" + "─"*80)
print("3. 이상치 & 주의 기업")
print("─"*80)

# 3a. DBO 유형별 분포
print(f"\nDBO 유형 분포 ({len(df_dbo_type)}개사):")
print(df_dbo_type['dbo_type'].value_counts().to_string())

# 3b. cik_mapping 이상치 플래그
flagged = m24[m24['flag'] != '']
if len(flagged) > 0:
    print(f"\ncik_mapping 이상치 플래그 ({len(flagged)}건):")
    print(flagged.groupby(['variable', 'flag']).size().reset_index(name='n').sort_values('n', ascending=False).to_string(index=False))

# 3c. DBO anomaly
anomaly_df = df_dbo_type[df_dbo_type['anomaly'] != '']
if len(anomaly_df) > 0:
    print(f"\nDBO anomaly ({len(anomaly_df)}건):")
    print(anomaly_df['anomaly'].value_counts().to_string())

# 3d. entity-only CIK (모든 변수가 entity)
entity_only_ciks = []
for cik in m24['CIK'].unique():
    cik_eids = m24[m24['CIK'] == cik]['ELEMENT_ID']
    if cik_eids.str.startswith('entity').all():
        entity_only_ciks.append(cik)
print(f"\nEntity-only CIK (전 변수 entity): {len(entity_only_ciks)}개")

# 3e. DBO도 NetDBO도 없는 CIK
has_dbo = set(m24[m24['variable'] == 'DBO']['CIK'])
has_net = set(m24[m24['variable'] == 'NetDBO']['CIK'])
no_dbo = set(m24['CIK']) - has_dbo - has_net
print(f"DBO/NetDBO 모두 없는 CIK: {len(no_dbo)}개")

# --- 4. 추출 전략 권고안 ---
print("\n" + "─"*80)
print("4. 추출 전략 권고안")
print("─"*80)
print("""
[우선순위 매칭 로직]
  1. ELEMENT_ID: ifrs-full_ > dart_ > entity{CIK}_
  2. UNIT_ID: 금액=KRW, 비율=PURE (불일치 행 제외)
  3. CONTEXT_ID: Consolidated > Default > Separate, CFY > PFY

[DBO 총액/순액 판별]
  1순위: IFRS DBO + PA 별도 존재 → 총액분리 확정
  2순위: Dimension축 (NetDefinedBenefitLiabilityAssetAxis) → 총액 Member 선택
  3순위: cal.tsv WEIGHT 검증 → 순액 여부 확인
  4순위: 한글 레이블 ('확정급여채무'=총액, '퇴직급여부채'=순액 추정)
  5순위: DBO 매핑값과 IFRS DBO 교차 비교 (±5% 내 일치)

[커버리지 한계]
  - 할인율/임금상승률/듀레이션: DBO의 15~20%만 보유 (주석 XBRL 태깅 미흡)
  - 민감도: 수집 불가 (1~2 CIK만 보유) → 제외
  - 총액 분리 가능: 전체의 약 3~4% (IFRS DBO+PA 보유 기업)
""")

# --- CSV 저장 ---
csv_report_path = BASE / 'pension_pattern_report.csv'
df_report.to_csv(csv_report_path, index=False, encoding='utf-8-sig')
print(f"\n저장: {csv_report_path}")

dbo_path = BASE / 'pension_dbo_type.csv'
df_dbo_type.to_csv(dbo_path, index=False, encoding='utf-8-sig')
print(f"저장: {dbo_path}")

print("\n" + "="*80)
print("탐색 완료 (v2)")
print("="*80)


XBRL 퇴직연금 부채 인덱스 — ELEMENT_ID 패턴 요약 리포트 (v2)

────────────────────────────────────────────────────────────────────────────────
1. 변수별 ELEMENT_ID 유형 분류
────────────────────────────────────────────────────────────────────────────────
                   변수                설명  ifrs-full  dart  entity  총_ELEMENT_ID  최대_CIK수 기대_UNIT                                                                              대표_ELEMENT_ID
                  DBO       확정급여채무 (총액)         39    10    3254          3303     1110     KRW                                                      dart_PostemploymentBenefitObligations
            PlanAsset       사외적립자산 공정가치         28     3    1049          1080      413     KRW                                                            ifrs-full_PlanAssetsAtFairValue
               NetDBO       순확정급여부채(자산)         33    13    1867          1913     1110     KRW                                                      dart_PostemploymentBenefitObligations
         DiscountRat

In [20]:
# =============================================================================
# Cell 9: 교차 검증 — 총액 분리 가능 기업 DBO/PA/Net 검증
# =============================================================================

import numpy as np

df_dbo = df_dbo_type.copy()

# 대상: IFRS DBO + PA 별도 보유 CIK
gross = df_dbo[(df_dbo['has_ifrs_dbo']==True) & (df_dbo['has_pa']==True)].copy()
print(f"IFRS DBO + PA 별도 보유 CIK: {len(gross)}개")
print(f"  dbo_type 분포: {gross['dbo_type'].value_counts().to_dict()}")
print()

# ── 1) PA 부호 이상 감지 ──
pa_neg = gross[gross['pa_val'] < 0]
print(f"1) PA 음수 CIK: {len(pa_neg)}개")
print(f"   → XBRL에서 자산을 음수로 기록하는 패턴 (NetDBO 롤업 내 차감항목)")
print(f"   → 부호 반전(abs) 적용하여 분석")
print()

# PA 부호 보정
gross['pa_adj'] = gross['pa_val'].abs()
gross['true_net'] = gross['ifrs_dbo_val'] - gross['pa_adj']

# DBO > 0 정상 케이스
valid = gross[gross['ifrs_dbo_val'] > 0].copy()
invalid = gross[gross['ifrs_dbo_val'] <= 0]
print(f"2) DBO 이상 CIK: {len(invalid)}개 (DBO ≤ 0)")
if len(invalid) > 0:
    for _, r in invalid.iterrows():
        print(f"   CIK={r['CIK']}: DBO={r['ifrs_dbo_val']/1e8:.1f}억, PA={r['pa_val']/1e8:.1f}억")
print()

# ── 3) NetDBO 매핑 정확도 ──
valid['net_eq_dbo'] = np.isclose(valid['net_val'], valid['ifrs_dbo_val'], rtol=0.01, equal_nan=False)
valid['net_eq_true'] = np.isclose(valid['net_val'], valid['true_net'], rtol=0.05, equal_nan=False)

n_eq_dbo = valid['net_eq_dbo'].sum()
n_eq_true = valid['net_eq_true'].sum()
n_other = len(valid) - n_eq_dbo - n_eq_true + valid[valid['net_eq_dbo'] & valid['net_eq_true']].shape[0]

print(f"3) NetDBO 매핑값 검증 ({len(valid)}개):")
print(f"   net_val ≈ DBO (총액과 동일 EID 매핑):   {n_eq_dbo}개")
print(f"   net_val ≈ DBO-PA (순액 정확):           {n_eq_true}개")
print(f"   기타 (불일치):                          {len(valid) - n_eq_dbo - n_eq_true}개")
print()

# ── 4) 적립비율 분석 ──
valid['funding_ratio'] = (valid['pa_adj'] / valid['ifrs_dbo_val'] * 100).round(1)
valid['dbo_억'] = (valid['ifrs_dbo_val'] / 1e8).round(1)
valid['pa_억'] = (valid['pa_adj'] / 1e8).round(1)
valid['net_억'] = (valid['true_net'] / 1e8).round(1)

print(f"4) 적립비율 분석 ({len(valid)}개, PA 부호보정):")
print(f"   평균: {valid['funding_ratio'].mean():.1f}%")
print(f"   중앙값: {valid['funding_ratio'].median():.1f}%")
print(f"   적립초과 (PA>DBO): {(valid['funding_ratio']>100).sum()}개")
print(f"   적립부족 (PA<DBO): {(valid['funding_ratio']<=100).sum()}개")
print()

bins = [0, 25, 50, 75, 100, 125, 150, 200, float('inf')]
labels = ['~25%','25~50%','50~75%','75~100%','100~125%','125~150%','150~200%','200%+']
valid['fr_bin'] = pd.cut(valid['funding_ratio'], bins=bins, labels=labels)
print("   적립비율 분포:")
for label in labels:
    n = (valid['fr_bin'] == label).sum()
    bar = '█' * (n // 3) if n > 0 else ''
    print(f"     {label:>10}: {n:4d} {bar}")
print()

# 가중평균
total_dbo = valid['dbo_억'].sum()
total_pa = valid['pa_억'].sum()
print(f"5) 가중평균 적립비율 (DBO 규모 가중):")
print(f"   총 DBO: {total_dbo:,.0f}억원")
print(f"   총 PA:  {total_pa:,.0f}억원")
print(f"   가중평균 적립비율: {total_pa/total_dbo*100:.1f}%")
print(f"   총 순부채(DBO-PA): {total_dbo - total_pa:,.0f}억원")
print()

# ── 6) 순부채 상위 20 ──
print("6) 순부채 상위 20 (억원):")
print(f"   {'CIK':>10} {'DBO':>12} {'PA':>12} {'순부채':>12} {'적립비율':>8}")
print(f"   {'-'*10} {'-'*12} {'-'*12} {'-'*12} {'-'*8}")
top20 = valid.nlargest(20, 'true_net')
for _, r in top20.iterrows():
    print(f"   {r['CIK']:>10} {r['dbo_억']:>11,.1f} {r['pa_억']:>11,.1f} {r['net_억']:>11,.1f} {r['funding_ratio']:>7.1f}%")

print()
print(f"{'='*60}")
print(f"교차 검증 완료: {len(valid)}개 총액분리 CIK 확인")

IFRS DBO + PA 별도 보유 CIK: 355개
  dbo_type 분포: {'총액(DBO매핑≈IFRS_DBO)': 336, '순액(DBO매핑=Net)': 13, '총액분리': 6}

1) PA 음수 CIK: 29개
   → XBRL에서 자산을 음수로 기록하는 패턴 (NetDBO 롤업 내 차감항목)
   → 부호 반전(abs) 적용하여 분석

2) DBO 이상 CIK: 9개 (DBO ≤ 0)
   CIK=00116949: DBO=-1991.8억, PA=-2325.8억
   CIK=00125150: DBO=-850.3억, PA=-933.6억
   CIK=00125743: DBO=-378.5억, PA=-271.8억
   CIK=00138260: DBO=-28.7억, PA=28.4억
   CIK=00148939: DBO=0.0억, PA=8.3억
   CIK=00150244: DBO=-5772.2억, PA=4507.5억
   CIK=00152686: DBO=-490.6억, PA=-356.4억
   CIK=00171265: DBO=-1857.5억, PA=1911.2억
   CIK=00172291: DBO=0.0억, PA=-826.6억

3) NetDBO 매핑값 검증 (346개):
   net_val ≈ DBO (총액과 동일 EID 매핑):   49개
   net_val ≈ DBO-PA (순액 정확):           119개
   기타 (불일치):                          178개

4) 적립비율 분석 (346개, PA 부호보정):
   평균: 143.9%
   중앙값: 101.2%
   적립초과 (PA>DBO): 177개
   적립부족 (PA<DBO): 169개

   적립비율 분포:
           ~25%:   17 █████
         25~50%:   20 ██████
         50~75%:   26 ████████
        75~100%:  101 █████████████████████████████████
 

In [21]:
# =============================================================================
# Cell 10: 2023→2024 CIK 변동 분석 (확정급여→확정기여 전환 / 상장폐지 식별)
# =============================================================================

# 매핑 기준 CIK 집합
cik_map_2024 = set(df_mapping[df_mapping['quarter'] == '2024_4Q']['CIK'].unique())
cik_map_2023 = set(df_mapping[df_mapping['quarter'] == '2023_4Q']['CIK'].unique())

only_2023 = sorted(cik_map_2023 - cik_map_2024)   # 2023에만 존재 (탈락)
only_2024 = sorted(cik_map_2024 - cik_map_2023)   # 2024에만 등장 (신규)
both = cik_map_2023 & cik_map_2024

print(f"2024_4Q 매핑 CIK: {len(cik_map_2024)}개")
print(f"2023_4Q 매핑 CIK: {len(cik_map_2023)}개")
print(f"양쪽 모두: {len(both)}개")
print(f"2023에만 (탈락): {len(only_2023)}개")
print(f"2024에만 (신규): {len(only_2024)}개")
print()

# sub.tsv 기준 제출 여부 확인 (Cell 1의 subs dict 사용)
sub_2024_ciks = set(subs['2024_4Q']['CIK'].unique())
sub_2023_ciks = set(subs['2023_4Q']['CIK'].unique())

in_sub_2024 = set(only_2023) & sub_2024_ciks      # 제출은 했지만 퇴직연금 매핑 탈락
not_in_sub_2024 = set(only_2023) - sub_2024_ciks   # 아예 미제출

print(f"{'='*60}")
print(f"탈락 {len(only_2023)}개 CIK 분류")
print(f"{'='*60}")
print(f"  A) 2024 미제출 (상장폐지/합병): {len(not_in_sub_2024)}개")
print(f"  B) 2024 제출 (태깅 중단/변경):  {len(in_sub_2024)}개")
print()

# 탈락 CIK의 DBO 규모
df_2023_only = df_mapping[
    (df_mapping['quarter'] == '2023_4Q') &
    (df_mapping['CIK'].isin(only_2023)) &
    (df_mapping['variable'] == 'DBO')
].copy()
df_2023_only['VALUE_억'] = pd.to_numeric(df_2023_only['sample_VALUE'], errors='coerce') / 1e8
df_2023_only['상태'] = df_2023_only['CIK'].apply(
    lambda x: 'A)미제출' if x in not_in_sub_2024 else 'B)태깅변경'
)

print("상태별 DBO 합계:")
for g, gdf in df_2023_only.groupby('상태'):
    print(f"  {g}: {len(gdf)}개 CIK, DBO합계 {gdf['VALUE_억'].sum():,.0f}억원")
print()

# DBO 10억 이상 탈락 기업 상세
big = df_2023_only[df_2023_only['VALUE_억'] >= 10].sort_values('VALUE_억', ascending=False)
print(f"DBO 10억+ 탈락 기업 ({len(big)}개):")
print(f"  {'CIK':>10} {'DBO(억)':>10} {'상태':>12} {'레이블'}")
print(f"  {'-'*10} {'-'*10} {'-'*12} {'-'*20}")
for _, r in big.iterrows():
    print(f"  {r['CIK']:>10} {r['VALUE_억']:>10,.1f} {r['상태']:>12} {r['ko_label']}")
print()

# DBO = 0인 탈락 CIK (확정기여 전환 가능성 높음)
zero_dbo = df_2023_only[df_2023_only['VALUE_억'] == 0]
print(f"DBO=0 탈락 CIK: {len(zero_dbo)}개 (확정기여 전환 또는 퇴직급여 소멸)")
print()

# ── 신규 등장 CIK ──
print(f"{'='*60}")
print(f"신규 {len(only_2024)}개 CIK")
print(f"{'='*60}")
df_2024_new = df_mapping[
    (df_mapping['quarter'] == '2024_4Q') &
    (df_mapping['CIK'].isin(only_2024)) &
    (df_mapping['variable'] == 'DBO')
].copy()
df_2024_new['VALUE_억'] = pd.to_numeric(df_2024_new['sample_VALUE'], errors='coerce') / 1e8
df_2024_new['기존상장'] = df_2024_new['CIK'].apply(lambda x: x in sub_2023_ciks)

n_existing = df_2024_new['기존상장'].sum()
n_ipo = len(df_2024_new) - n_existing

print(f"  DBO 보유 신규 CIK: {len(df_2024_new)}개")
print(f"    기존 상장사 (XBRL 태깅 신규/변경): {n_existing}개")
print(f"    신규 상장 (IPO 등): {n_ipo}개")
print(f"  신규 DBO 합계: {df_2024_new['VALUE_억'].sum():,.0f}억원")
print()

# 대형 신규
big_new = df_2024_new[df_2024_new['VALUE_억'] >= 10].sort_values('VALUE_억', ascending=False)
if len(big_new) > 0:
    print(f"DBO 10억+ 신규 기업 ({len(big_new)}개):")
    for _, r in big_new.head(20).iterrows():
        tag = '기존→태깅추가' if r['기존상장'] else 'IPO'
        print(f"  CIK={r['CIK']} DBO={r['VALUE_억']:,.1f}억 [{r['ko_label']}] — {tag}")

print()
print(f"{'='*60}")
print(f"변동 분석 완료")

2024_4Q 매핑 CIK: 2303개
2023_4Q 매핑 CIK: 2204개
양쪽 모두: 2109개
2023에만 (탈락): 95개
2024에만 (신규): 194개

탈락 95개 CIK 분류
  A) 2024 미제출 (상장폐지/합병): 54개
  B) 2024 제출 (태깅 중단/변경):  41개

상태별 DBO 합계:
  A)미제출: 52개 CIK, DBO합계 9,731억원
  B)태깅변경: 25개 CIK, DBO합계 69억원

DBO 10억+ 탈락 기업 (23개):
         CIK     DBO(억)           상태 레이블
  ---------- ---------- ------------ --------------------
    00113410    3,232.3        A)미제출 확정급여채무, 현재가치
    00164609    2,770.0        A)미제출 확정급여채무, 현재가치
    00158307    1,253.5        A)미제출 확정급여채무, 현재가치
    00360595    1,197.2        A)미제출 확정급여채무, 현재가치
    00148896      365.4        A)미제출 확정급여채무, 현재가치
    00107987      362.3        A)미제출 확정급여채무의현재가치
    00159810      108.6        A)미제출 순확정급여부채
    01350869       69.4        A)미제출 순확정급여부채
    01077577       65.1        A)미제출 순확정급여채무
    00620868       62.2        A)미제출 확정급여부채
    00676724       23.6        A)미제출 순확정급여부채
    00267058       23.1        A)미제출 순확정급여부채
    00922304       17.9        A)미제출 순확정급여부채
    01633513       17.4 